# FoX A100 experiment: worst-case generalization along growing radii

This notebook trains the restricted answer-supervised model on **N=128 keys, width d=4096,
training lag R=2, 200,000 updates, three paired seeds and four optimizer/gate arms**.
It measures both ordinary finite-prefix prediction curves and certified brackets for the
worst-case error **Eₜ(r)** at fixed and growing radii. All executable source and tests are embedded.

In Colab, choose **Runtime → Change runtime type → A100 GPU**, then run all cells.
The notebook checks the hardware you received; it does not provision or guarantee an A100.
GPU availability and runtime lifetimes vary, so the default stores resumable checkpoints on Drive.
See the [official Colab resource and runtime FAQ](https://research.google.com/colaboratory/faq.html#resource-limits)
and its [Drive guidance](https://research.google.com/colaboratory/faq.html#drive-mount).

The execution order is: software checks, a short **seed-0 benchmark at the requested model size**,
then seed 0's four full arms, followed by the remaining seeds. The measured benchmark prints a
rough compute estimate before the full run. Change `SEEDS=(0,)` for an initial single-seed study;
the full default is `(0,1,2)`. `SMOKE=True` runs a tiny CPU software check, not an A100 experiment.

The longer budget does not guarantee that asymptotic behavior has appeared. The report retains
loss, gaps, optimizer clocks, actual growing radii, interval validity, and stopped-run statuses.


## What Eₜ(r) means, and what we can bound

\[
E_t(r)=\sup_{\substack{\text{finite record streams }x\\
\text{latest target lag}(x)\le r}}
\bigl(1-P_t(\text{correct answer}\mid x)\bigr).
\]

This is worst-case **answer error over all finite streams**, not an average over test examples,
and not the epsilon in Adam's denominator. We save a bracket
\(L_t(r)\le E_t(r)\le U_t(r)\):

- The lower endpoint is an analytic infinite-prefix witness limit. Every finite prefix is an
  allowed stream, so the limit lower-bounds the supremum.
- The upper endpoint is a uniform certificate over arbitrary finite streams, conditional on its
  recorded matching-gap, row-norm, scale and sign conditions. When those conditions fail, we mark
  the certificate invalid and show the trivial upper bound **E≤1** explicitly.
- Both endpoints are stored and plotted in log-error space. We do not compute tiny errors by
  subtracting a rounded prediction probability from one.

Fixed probes examine Eₜ(r) at fixed r. Moving probes examine Eₜ(rₜ), and plot the actual integer rₜ
alongside the error bounds. Clock probes use `floor(1+c*S)` for learned retrieval and
`floor(1+c*S**2)` for frozen retrieval. These are prescribed experiments, not growth-rate claims
for every optimizer. Adaptive and acquired-reference probes use the observed matching gaps.
The largest radius certified at each error threshold is computed from the uniform bound with
**no fixed evaluation-lag cap**.

For frozen SGD, the polynomial clock probes are stress tests: the ordinary-answer theorem does
not provide a sharp time law for its expanding range. Pointer-supervised time laws are not used here.

To support a growing-range conclusion, inspect both a decreasing uniform upper error and an
increasing actual radius. A small error at a radius that stays bounded does not establish
generalization along rₜ→∞, and a finite trajectory cannot prove either limiting claim.


## Dataset, interventions, and theoretical scope

Every ordered pair a≠b is included: **K=N(N−1)=16,256 pair identities** at N=128.
For each pair we train the paper's overwrite sequence
`(a,-1),(a,-1),(b,-1),(a,+1),?a` and recall sequence
`(a,+1),(b,-1)×(R−1),?a`, together with both global value complements.
Two one-record calibration sequences are included per key. Thus the complete weighted population
has **4K+2N serialized sequences**. Lags count records, newest record at lag 1.
The overwrite/recall/calibration weights are 0.2/0.2/0.6.

The four arms are learned retrieval / SGD, learned retrieval / Adam, frozen retrieval / SGD,
and frozen retrieval / Adam. Frozen retrieval holds **h=h₀** while the local binding gate remains
trainable. Raw Q/K table draws and scalar initialization are paired within seed. Three acquisition
updates precede continuation; Adam keeps its zero-initialized moment history without resets.

At R=2, the paper predicts a learned-SGD worst-case cutoff at lag 4, an expanding learned-Adam
range, and expanding frozen-retrieval ranges under the sufficient protocol, with h₀>log(2).
The frozen-Adam range has a quadratic sufficient clock scaling. The practical polynomial schedule
here is an empirical protocol, not the theorem's very conservative existence construction.
R>2 is available as an extension experiment, with additional hypotheses and preparation caveats.

N=128,d=4096 satisfies the stated Gaussian dimension lower bound at failure budget 0.05.
That verifies this one dimension condition only; it does not certify every initialization, rate,
settling, cone or continuation hypothesis. Increasing compute alone is not proof of asymptotics.


In [ ]:
# USER SETTINGS — keep RUN_TAG and settings unchanged to resume interrupted work.
from pathlib import Path
import importlib.util
import os
import sys
import tempfile

SMOKE = False
N, WIDTH, R, STEPS = 128, 4096, 2, 200000
SEEDS = (0, 1, 2)                 # Use (0,) for a first full seed.
H0 = 1.0
MOMENT_PRESET = "paper_short_memory"  # Or "standard_memory"; uses a separate run folder.
RUN_TAG = "fox_a100_asymptotics_v1"
ALLOW_OTHER_GPU = False
USE_DRIVE = True
RUN_QA = True
RUN_BENCHMARK = True
RUN_TRAINING = True
DOWNLOAD_AT_END = False           # Results already persist on Drive by default.
COMPILE = False                   # Experimental compilation is deliberately opt-in.
RESUME = True
LOG_EVERY = 1000
CHECKPOINT_EVERY = 5000
BENCHMARK_STEPS, BENCHMARK_WARMUP = 30, 5
MAX_BRANCH_SECONDS = 24 * 3600
EVAL_LAGS = (1, 2, 3, 4, 5, 8, 16, 32, 64, 128, 256, 512, 1024)
PREFIXES = (0, 64)
FIXED_LAGS = (1, 2, 3, 4, 5, 8, 16, 32, 64)
THETA_VALUES = (0.25, 0.5, 0.75)
CLOCK_COEFFICIENTS = (0.001, 0.003, 0.01, 0.03, 0.1)
ERROR_TARGETS = (0.1, 0.01, 0.001)

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "fox_a100_mpl"))
import numpy as np
import torch
from IPython.display import display, Image, FileLink, Markdown

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except (ModuleNotFoundError, ValueError):
    IN_COLAB = False

if MOMENT_PRESET not in {"paper_short_memory", "standard_memory"}:
    raise ValueError("Choose paper_short_memory or standard_memory.")
BETA1, BETA2 = {"paper_short_memory": (0.1, 0.1), "standard_memory": (0.9, 0.999)}[MOMENT_PRESET]
assert BETA1**2 < BETA2
if SMOKE:
    N, WIDTH, STEPS, SEEDS = 4, 64, 20, (0,)
    EVAL_LAGS, PREFIXES = (1, 2, 3, 4, 5, 8), (0, 4)
    FIXED_LAGS, THETA_VALUES, CLOCK_COEFFICIENTS = (1, 2, 4, 8), (0.5,), (0.01,)
    LOG_EVERY, CHECKPOINT_EVERY = 10, 10
    BENCHMARK_STEPS, BENCHMARK_WARMUP = 3, 1
    USE_DRIVE, DOWNLOAD_AT_END = False, False
    DEVICE = "cpu"
else:
    if not torch.cuda.is_available():
        raise RuntimeError("This profile requires a CUDA GPU. Select an A100 runtime in Colab, or set SMOKE=True for a CPU check.")
    DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)
    if "A100" not in GPU_NAME.upper() and not ALLOW_OTHER_GPU:
        raise RuntimeError(f"Received {GPU_NAME}, not an A100. Select an A100 or explicitly set ALLOW_OTHER_GPU=True.")
    if "A100" not in GPU_NAME.upper():
        print("Explicit override: running the A100 profile on", GPU_NAME)

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

if USE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Drive mounting is supported here in Colab. Set USE_DRIVE=False for another CUDA environment.")
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_PARENT = Path("/content/drive/MyDrive/fox_a100_runs")
else:
    OUTPUT_PARENT = Path(os.environ.get("FOX_A100_OUTPUT_PARENT", str(Path.cwd() / "fox_a100_runs")))
EFFECTIVE_RUN_TAG = RUN_TAG + "_" + MOMENT_PRESET + ("_smoke" if SMOKE else "")
OUT_DIR = OUTPUT_PARENT / EFFECTIVE_RUN_TAG
CODE_DIR = Path(tempfile.mkdtemp(prefix="fox_a100_source_"))
SOURCES = {}
HARDWARE = dict(device=DEVICE, gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
                torch_version=torch.__version__, parameter_dtype="float64", tf32=False, mixed_precision=False)
print("Hardware:", HARDWARE)
print("Persistent output:", OUT_DIR)
print(f"Plan: seed {SEEDS[0]} first, followed by {SEEDS[1:]}; {4*len(SEEDS)} arms × {STEPS:,} total updates.")
print(f"K={N*(N-1):,} pair identities; {4*N*(N-1)+2*N:,} complete serialized sequences.")
print("Moment preset:", MOMENT_PRESET, "| betas:", (BETA1, BETA2))


## Exact FP64 implementation and preflight checks

Parameters, signed-log gradients, and signed-log Adam moments remain float64. There is no BF16,
FP16, autocast, TF32, gradient clipping, moment reset or epsilon floor. Adam's prescribed epsilon
is outside the square root and anneals as sigma³ exp(−c t). Signed-log arithmetic retains very
small derivatives but still has finite precision. Independent native-autodiff checks, witness
bounds and resume tests run before training.

The default `paper_short_memory` betas `(0.1,0.1)` follow the existing mechanism experiment's
short-memory setting. They are selected to study moment tracking of changing gradients, not by
long-lag validation performance. The earlier baseline used `(0.9,0.999)`; retain that comparison
with `standard_memory`. Both satisfy beta1²<beta2, and both initialize all moments at zero.
Their separate run folders prevent accidental mixing or overwriting.
Changing the betas affects both acquisition and later moment dynamics, so this comparison does
not isolate the continuation-memory effect while holding the acquired key geometry fixed.


In [ ]:
SOURCES['core'] = '"""Restricted ordinary-answer FoX experiment, implemented in float64.\n\nThe analytic backend evaluates the exact finite data objective. Signed-log\ngradients and Adam buffers retain very small derivatives without substituting\npointer supervision, sign descent, gradient clipping, or an epsilon floor.\n``dense_stream`` independently materializes both attention heads for auditing.\n"""\nfrom dataclasses import asdict, dataclass\nimport json\nimport math\nfrom pathlib import Path\n\nimport torch\nfrom torch.nn import functional as F\n\n\n@dataclass\nclass Config:\n    n: int = 8\n    d: int = 2048\n    R: int = 2\n    steps: int = 12000\n    seed: int = 0\n    device: str = "cpu"\n    sigma: float = 1e-6\n    gamma: float = 1 / 16\n    c0: float = 1.\n    D: float = 0.\n    kcal: float = .6\n    wo: float = .2\n    wc: float = .2\n    q0: float = .5\n    u0: float = 1.7\n    h0: float = 1.\n    x0: float | None = None\n    gate_mode: str = "learned"\n    am: float = 1.\n    ag: float = 1.5\n    ax: float = .05\n    aw: float = .1\n    beta1: float = .9\n    beta2: float = .999\n    eps_decay: float = .1\n    lr_adam: float = .015\n    lr_sgd: float = 5.\n    offset: float = 1000.\n    power: float = .75\n    table_lr: float = 1e-9\n    table_power: float = 2.\n    pair_batch: int = 4096\n    batch_growth: float = .5\n    log_points: int = 80\n    eval_max_lag: int = 128\n    max_seconds: float = 3600.\n\n    def __post_init__(self):\n        for name, minimum in (("n", 2), ("d", 1), ("R", 2), ("steps", 0),\n                              ("pair_batch", 1), ("log_points", 2), ("eval_max_lag", 1)):\n            value = getattr(self, name)\n            if isinstance(value, bool) or not isinstance(value, int) or value < minimum:\n                raise ValueError(f"{name} must be an integer >= {minimum}")\n        if self.gate_mode not in ("learned", "retrieval_frozen", "both_frozen"):\n            raise ValueError("gate_mode must be learned, retrieval_frozen, or both_frozen")\n        positive = ("sigma", "gamma", "c0", "q0", "u0", "h0", "kcal", "wo", "wc",\n                    "am", "ag", "ax", "aw", "lr_adam", "lr_sgd", "offset", "table_lr")\n        if any(not math.isfinite(getattr(self, name)) or getattr(self, name) <= 0 for name in positive):\n            raise ValueError("Initialization scales, mixture weights, and learning rates must be positive and finite")\n        if not self.kcal > .5 or not math.isclose(self.kcal + self.wo + self.wc, 1., abs_tol=1e-12):\n            raise ValueError("Require kcal > 1/2 and kcal + wo + wc = 1")\n        if not 0 <= self.beta1 < 1 or not self.beta1 ** 2 < self.beta2 < 1:\n            raise ValueError("Adam requires 0 <= beta1 < 1 and beta1**2 < beta2 < 1")\n        if not 2 / 3 < self.power <= 1 or not self.table_power > 1:\n            raise ValueError("Use scalar power in (2/3, 1] and summable table power > 1")\n        if self.eps_decay < 0 or self.batch_growth < 0:\n            raise ValueError("epsilon decay and batch growth must be nonnegative")\n        if self.x0 is None:\n            self.x0 = self.h0 + math.log(-math.expm1(-self.h0))\n        if not math.isfinite(self.x0) or not math.isfinite(self.D):\n            raise ValueError("x0 and D must be finite")\n        effective_h = max(self.x0, 0.) + math.log1p(math.exp(-abs(self.x0)))\n        if not math.isclose(effective_h, self.h0, rel_tol=1e-12, abs_tol=1e-12):\n            raise ValueError("x0 must equal inverse_softplus(h0); use replace(cfg, h0=new_h, x0=None) to change the slope")\n\n\ndef frozen_scalar_indices(config):\n    return {"learned": (), "retrieval_frozen": (4,), "both_frozen": (2, 3, 4)}[config.gate_mode]\n\n\ndef active_parameter_count(config):\n    return 2 * config.n * config.d + 6 - len(frozen_scalar_indices(config))\n\n\ndef slog(x):\n    return torch.sign(x), torch.log(torch.abs(x))\n\n\ndef sadd(s1, l1, s2, l2):\n    """Signed log addition; exact cancellation has sign zero and log -inf."""\n    same = s1 == s2\n    hi = torch.maximum(l1, l2)\n    diff = torch.abs(l1 - l2)\n    difference = hi + torch.log(-torch.expm1(-diff))\n    logabs = torch.where(same, torch.logaddexp(l1, l2), difference)\n    sign = torch.where(same, s1, torch.where(l1 > l2, s1, s2))\n    zero = ((l1 == l2) & ~same) | (torch.isneginf(l1) & torch.isneginf(l2))\n    return torch.where(zero, 0., sign), torch.where(zero, -torch.inf, logabs)\n\n\ndef ssum(signs, logs, dim=None):\n    if dim is None:\n        signs, logs, dim = signs.reshape(-1), logs.reshape(-1), 0\n    pos = torch.logsumexp(torch.where(signs > 0, logs, -torch.inf), dim)\n    neg = torch.logsumexp(torch.where(signs < 0, logs, -torch.inf), dim)\n    return sadd(torch.ones_like(pos), pos, -torch.ones_like(neg), neg)\n\n\ndef logsoftplus(x):\n    # For x < -30, the relative difference from exp(x) is < 5e-14.\n    return torch.where(x < -30, x, torch.log(F.softplus(x)))\n\n\ndef quantities_from_theta(config, theta, detach_frozen=False):\n    q, p, u, v, x, w = theta.unbind()\n    if detach_frozen:\n        if config.gate_mode == "both_frozen":\n            u, v = u.detach(), v.detach()\n        if config.gate_mode != "learned":\n            x = x.detach()\n    m, g, h = config.c0 * q * p, F.softplus(u * v), F.softplus(x)\n    lrho = F.logsigmoid(config.D - 2 * g)\n    rho = torch.exp(lrho)\n    return q, p, u, v, x, w, m, g, h, rho, lrho\n\n\ndef recall_terms(z, m, h, rho, lag):\n    """One term per wrong record, in target-relative log-weight units."""\n    if lag < 2:\n        raise ValueError("Recall training lag must be >= 2")\n    first = -z * m * (1 - rho) + h\n    if lag == 2:\n        return first.unsqueeze(-1)\n    offsets = torch.arange(2, lag, dtype=z.dtype, device=z.device)\n    pure = (-z * m).unsqueeze(-1) + offsets * h\n    return torch.cat((first.unsqueeze(-1), pure), dim=-1)\n\n\ndef _pair_log_mass(z, counts, validate=True):\n    if counts is None:\n        return torch.full_like(z, -math.log(z.numel()))\n    counts = torch.as_tensor(counts, device=z.device, dtype=z.dtype)\n    if validate:\n        if counts.shape != z.shape or bool((counts < 0).any()) or not bool(counts.sum() > 0):\n            raise ValueError("counts must be a nonnegative vector with positive total, one entry per ordered pair")\n        if not bool(torch.isfinite(counts).all()):\n            raise ValueError("counts must be finite")\n    return torch.log(counts) - torch.log(counts.sum())\n\n\nclass Model:\n    def __init__(self, cfg):\n        self.cfg = cfg\n        generator = torch.Generator(device=cfg.device).manual_seed(cfg.seed)\n        self.Q = torch.randn(cfg.n, cfg.d, generator=generator, device=cfg.device, dtype=torch.float64) * cfg.sigma\n        self.K = torch.randn(cfg.n, cfg.d, generator=generator, device=cfg.device, dtype=torch.float64) * cfg.sigma\n        self.theta = torch.tensor([cfg.q0, cfg.q0, cfg.u0, cfg.u0, cfg.x0, 0.],\n                                  device=cfg.device, dtype=torch.float64)\n        self.mask = ~torch.eye(cfg.n, device=cfg.device, dtype=torch.bool)\n        self.aa, self.bb = torch.where(self.mask)\n        self.frozen_scalar_indices = frozen_scalar_indices(cfg)\n\n    def params(self):\n        return [self.Q, self.K, self.theta]\n\n    def gaps(self):\n        scores = self.Q @ self.K.T\n        return scores.diag()[:, None] - scores\n\n    def quantities(self):\n        return quantities_from_theta(self.cfg, self.theta)\n\n    @property\n    def active_parameter_count(self):\n        return active_parameter_count(self.cfg)\n\n    def rows(self, z):\n        q, p, u, v, x, w, m, g, h, rho, lrho = self.quantities()\n        terms = torch.stack((z * m * rho - 2 * h, z * m * rho - 3 * h,\n                             -z * m * (1 - 2 * rho) - h), dim=-1)\n        lo = torch.logsumexp(terms, dim=-1)\n        lc = torch.logsumexp(recall_terms(z, m, h, rho, self.cfg.R), dim=-1)\n        return terms, lo, lc, -torch.tanh(lo / 2), -torch.tanh(lc / 2)\n\n    @torch.no_grad()\n    def gradients(self, counts=None, validate_counts=True):\n        """Exact ordinary-loss derivatives in sign/log-absolute representation.\n\n        Pair counts give a stratified estimator: calibration is exact and both\n        overwrite and recall losses are evaluated for every sampled pair.\n        ``validate_counts=False`` is only for internally generated multinomial\n        counts. It removes host synchronizations from the trusted GPU path.\n        """\n        c = self.cfg\n        q, p, u, v, x, w, m, g, h, rho, lrho = self.quantities()\n        z = self.gaps()[self.aa, self.bb]\n        terms, lo, lc, so, sc = self.rows(z)\n        rterms = recall_terms(z, m, h, rho, c.R)\n        lp = _pair_log_mass(z, counts, validate=validate_counts)\n        ltw = torch.log(2 * torch.abs(w))\n        common_o = lp + math.log(c.wo) + ltw + F.logsigmoid(-w * so) - 2 * F.softplus(lo)\n        common_c = lp + math.log(c.wc) + ltw + F.logsigmoid(-w * sc) - 2 * F.softplus(lc)\n        la, lb, le = (common_o[:, None] + terms).unbind(-1)\n        recall_pressure = common_c[:, None] + rterms\n        lf = recall_pressure[:, 0]\n        # P is the derivative with respect to M=z*m, divided by sign(w).\n        logs_p = torch.cat((torch.stack((la + lrho, lb + lrho,\n                                        le + torch.log(torch.abs(1 - 2 * rho)),\n                                        lf + torch.log1p(-rho)), dim=-1),\n                            recall_pressure[:, 1:]), dim=-1)\n        signs_p = -torch.ones_like(logs_p)\n        signs_p[:, :2] = 1.\n        signs_p[:, 2] = -torch.sign(1 - 2 * rho)\n        ps, pl = ssum(signs_p, logs_p, dim=-1)\n        ps = ps * torch.sign(w)\n        dzs, dzl = ps * torch.sign(m), pl + torch.log(torch.abs(m))\n        shift = torch.max(dzl)\n        shift = torch.where(torch.isfinite(shift), shift, torch.zeros_like(shift))\n        weights = torch.zeros(c.n, c.n, device=z.device, dtype=z.dtype)\n        weights[self.aa, self.bb] = dzs * torch.exp(dzl - shift)\n        score_grad = -weights\n        score_grad.diagonal().add_(weights.sum(dim=1))\n        sq, lq = slog(score_grad @ self.K)\n        sk, lk = slog(score_grad.T @ self.Q)\n        sm, lm = ssum(ps * torch.sign(z), pl + torch.log(torch.abs(z)))\n        rec_offsets = torch.arange(1, c.R, dtype=z.dtype, device=z.device)\n        logs_h = torch.cat((torch.stack((la + math.log(2), lb + math.log(3), le), dim=-1),\n                            recall_pressure + rec_offsets.log()), dim=-1)\n        signs_h = torch.ones_like(logs_h)\n        signs_h[:, :3] = -1.\n        sh, lh = ssum(signs_h * torch.sign(w), logs_h)\n        # Only the first recall distractor\'s binder leaks the queried key.\n        binder_pressure = torch.logsumexp(torch.stack((la, lb, le + math.log(2), lf), dim=-1), dim=-1)\n        sg, lg = ssum(-torch.sign(z * m * w), binder_pressure + torch.log(torch.abs(z * m))\n                     + math.log(2) + lrho + torch.log1p(-rho))\n        sw, lw = ssum(torch.cat((-torch.ones(1, device=z.device, dtype=z.dtype), -torch.sign(so), -torch.sign(sc))),\n                       torch.cat(((math.log(c.kcal) + F.logsigmoid(-w)).reshape(1),\n                                  lp + math.log(c.wo) + torch.log(torch.abs(so)) + F.logsigmoid(-w * so),\n                                  lp + math.log(c.wc) + torch.log(torch.abs(sc)) + F.logsigmoid(-w * sc))))\n        st = torch.stack((sm * torch.sign(p), sm * torch.sign(q), sg * torch.sign(v),\n                          sg * torch.sign(u), sh, sw))\n        lt = torch.stack((lm + torch.log(torch.abs(c.c0 * p)), lm + torch.log(torch.abs(c.c0 * q)),\n                          lg + torch.log(torch.abs(v)) + F.logsigmoid(u * v),\n                          lg + torch.log(torch.abs(u)) + F.logsigmoid(u * v),\n                          lh + F.logsigmoid(x), lw))\n        if self.frozen_scalar_indices:\n            indices = list(self.frozen_scalar_indices)\n            st[indices], lt[indices] = 0., -torch.inf\n        logloss = torch.logsumexp(torch.cat(((math.log(c.kcal) + logsoftplus(-w)).reshape(1),\n                                             lp + math.log(c.wo) + logsoftplus(-w * so),\n                                             lp + math.log(c.wc) + logsoftplus(-w * sc))), dim=0)\n        finite = torch.isfinite(dzl)\n        high = torch.where(finite, dzl, -torch.inf).max()\n        low = torch.where(finite, dzl, torch.inf).min()\n        span = torch.where(finite.any(), high - low, torch.zeros_like(w))\n        return [(sq, lq + shift), (sk, lk + shift), (st, lt)], {"logloss": logloss, "table_log_span": span}\n\n\ndef native_loss(model, counts=None, params=None):\n    """Conventional PyTorch autograd loss, used as an independent derivative check.\n\n    This path may underflow at extreme checkpoints; it never substitutes a\n    log-gradient result when native arithmetic loses a derivative.\n    """\n    Q, K, theta = model.params() if params is None else params\n    c = model.cfg\n    q, p, u, v, x, w, m, g, h, rho, lrho = quantities_from_theta(c, theta, detach_frozen=True)\n    scores = Q @ K.T\n    z = (scores.diag()[:, None] - scores)[model.mask]\n    log_o = torch.logsumexp(torch.stack((z * m * rho - 2 * h, z * m * rho - 3 * h,\n                                       -z * m * (1 - 2 * rho) - h), dim=-1), dim=-1)\n    log_c = torch.logsumexp(recall_terms(z, m, h, rho, c.R), dim=-1)\n    loss = c.wo * F.softplus(w * torch.tanh(log_o / 2)) + c.wc * F.softplus(w * torch.tanh(log_c / 2))\n    return c.kcal * F.softplus(-w) + (loss * _pair_log_mass(z, counts).exp()).sum()\n\n\ndef native_gradients(model, counts=None):\n    with torch.enable_grad():\n        params = [p.detach().clone().requires_grad_(True) for p in model.params()]\n        loss = native_loss(model, counts, params)\n        gradients = [p.detach() for p in torch.autograd.grad(loss, params)]\n    return gradients, {"loss": loss.detach(), "logloss": loss.detach().log(),\n                       "native_loss_underflow": bool(loss.detach() == 0)}\n\n\ndef dense_stream(params, config, keys, values, query, label=1.):\n    """Literal two-head forward pass on serialized key/value record inputs.\n\n    Returns ordinary answer loss, record attention, and correct-answer logit.\n    The binder scores raw-token offsets 1 and 3. Retrieval uses slope h/2 per\n    raw token, equivalent to h per record; the query is never a candidate.\n    """\n    Q, K, theta = params\n    q, p, u, v, x, w = theta.unbind()\n    if config.gate_mode == "both_frozen":\n        u, v = u.detach(), v.detach()\n    if config.gate_mode != "learned":\n        x = x.detach()\n    g, h = F.softplus(u * v), F.softplus(x)\n    keys = torch.as_tensor(keys, device=Q.device, dtype=torch.long)\n    values = torch.as_tensor(values, device=Q.device, dtype=Q.dtype)\n    if keys.ndim != 1 or values.shape != keys.shape or keys.numel() < 1:\n        raise ValueError("A stream needs equally sized nonempty key/value vectors")\n    if bool(((keys < 0) | (keys >= config.n)).any()) or not 0 <= int(query) < config.n:\n        raise ValueError("Key IDs must be in the configured vocabulary")\n    if not bool(((values == -1) | (values == 1)).all()) or float(label) not in (-1., 1.):\n        raise ValueError("Values and answer label must be -1 or +1")\n    vectors = K[keys]\n    binding_weights = torch.stack((-g, config.D - 3 * g)).softmax(dim=0)\n    bound = torch.cat((vectors[:1], binding_weights[0] * vectors[1:] + binding_weights[1] * vectors[:-1]), dim=0)\n    raw_distances = 2 * torch.arange(keys.numel(), 0, -1, device=Q.device, dtype=Q.dtype)\n    scores = config.c0 * q * p * (bound @ Q[int(query)]) - (h / 2) * raw_distances\n    attention = scores.softmax(dim=0)\n    correct_logit = label * w * torch.sum(attention * values)\n    return F.softplus(-correct_logit), attention, correct_logit\n\n\ndef dense_objective(model, counts=None, params=None):\n    """Slow literal-stream objective, including both label complements."""\n    params = model.params() if params is None else params\n    c = model.cfg\n    rows = []\n    for a in range(c.n):\n        for b in range(c.n):\n            if a == b:\n                continue\n            o = sum(dense_stream(params, c, [a, a, b, a], [-y, -y, -y, y], a, y)[0] for y in (-1, 1)) / 2\n            rec = sum(dense_stream(params, c, [a] + [b] * (c.R - 1), [y] + [-y] * (c.R - 1), a, y)[0] for y in (-1, 1)) / 2\n            rows.append(c.wo * o + c.wc * rec)\n    rows = torch.stack(rows)\n    calibration = torch.stack([dense_stream(params, c, [a], [y], a, y)[0]\n                               for a in range(c.n) for y in (-1, 1)]).mean()\n    return c.kcal * calibration + (rows * _pair_log_mass(rows, counts).exp()).sum()\n\n\nclass LogAdam:\n    """Bias-corrected Adam with signed-log first and log second moments.\n\n    All buffers start at zero. Moment histories persist across acquisition and\n    continuation. Only storage arithmetic differs from conventional Adam.\n    """\n    def __init__(self, params, b1, b2):\n        if not 0 <= b1 < 1 or not b1 ** 2 < b2 < 1:\n            raise ValueError("Require 0 <= beta1 < 1 and beta1**2 < beta2 < 1")\n        self.b1, self.b2, self.t = b1, b2, 0\n        self.ms = [torch.zeros_like(p) for p in params]\n        self.ml = [torch.full_like(p, -torch.inf) for p in params]\n        self.vl = [torch.full_like(p, -torch.inf) for p in params]\n\n    @torch.no_grad()\n    def step(self, params, grads, rates, logeps):\n        self.t += 1\n        correction_m = math.log1p(-self.b1 ** self.t)\n        correction_v = math.log1p(-self.b2 ** self.t)\n        directions = []\n        for i, (parameter, (sign, logabs), rate) in enumerate(zip(params, grads, rates)):\n            self.ms[i], self.ml[i] = sadd(self.ms[i], self.ml[i] + (math.log(self.b1) if self.b1 else -math.inf),\n                                          sign, logabs + math.log1p(-self.b1))\n            self.vl[i] = torch.logaddexp(self.vl[i] + math.log(self.b2), 2 * logabs + math.log1p(-self.b2))\n            denominator = torch.logaddexp(.5 * (self.vl[i] - correction_v), torch.full_like(parameter, logeps))\n            direction = torch.where(self.ms[i] == 0, 0., self.ms[i] * torch.exp(self.ml[i] - correction_m - denominator))\n            parameter.add_(-rate * direction)\n            directions.append(direction)\n        return directions\n\n\n@torch.no_grad()\ndef gradient_step(model, grads, rates):\n    for parameter, (sign, logabs), rate in zip(model.params(), grads, rates):\n        parameter.add_(-rate * sign * torch.exp(logabs))\n\n\ndef _mask_rates(config, scalar_rates):\n    scalar_rates[list(frozen_scalar_indices(config))] = 0.\n    return scalar_rates\n\n\ndef rates_at(c, kind, t):\n    if kind not in ("sgd", "adam", "adam_annealed", "adam_fixed"):\n        raise ValueError("kind must be sgd, adam, adam_annealed, or adam_fixed")\n    base = (c.lr_adam if kind.startswith("adam") else c.lr_sgd) * (1 + t / c.offset) ** (-c.power)\n    table = c.table_lr * (1 + t / c.offset) ** (-c.table_power)\n    multipliers = _mask_rates(c, torch.tensor([c.am, c.am, c.ag, c.ag, c.ax, c.aw], device=c.device, dtype=torch.float64))\n    return base, [table, table, base * multipliers]\n\n\ndef acquisition_rates(c, kind):\n    """Three ordinary-loss updates with deterministic rates fixed before draws.\n\n    SGD rates use the exact zero-gap objective at this R and initialization.\n    These practical finite constants do not instantiate the proof\'s unspecified\n    sufficiently-small bounds or the extra R>2 basin preparation protocol.\n    """\n    if kind.startswith("adam"):\n        theta = _mask_rates(c, torch.tensor([1e-5] * 5 + [.005], device=c.device, dtype=torch.float64))\n        return [[c.gamma / math.sqrt(c.d)] * 2 + [theta.clone()] for _ in range(3)]\n    if kind != "sgd":\n        raise ValueError("Acquisition supports sgd or adam")\n    probe = Model(c)\n    probe.Q.zero_()\n    probe.K.zero_()\n    sw, lw = probe.gradients()[0][-1]\n    fw = float(sw[-1] * lw[-1].exp())\n    if not math.isfinite(fw) or fw >= 0:\n        raise ValueError(f"Initial decoder derivative must be negative; got {fw}")\n    probe.theta[-1] = .2\n    with torch.enable_grad():\n        z = torch.zeros((), device=c.device, dtype=torch.float64, requires_grad=True)\n        _, _, _, so, sc = probe.rows(z)\n        phi = c.wo * F.softplus(-.2 * so) + c.wc * F.softplus(-.2 * sc)\n        dstar = -float(torch.autograd.grad(phi, z)[0])\n    if not math.isfinite(dstar) or dstar <= 0:\n        raise ValueError(f"Acquisition requires a positive zero-gap matching signal; dstar={dstar}")\n    astar = dstar / (c.n - 1)\n    theta = _mask_rates(c, torch.tensor([1e-7] * 5 + [1e-4], device=c.device, dtype=torch.float64))\n    first = theta.clone()\n    first[-1] = .2 / (-fw)\n    return [[1e-7, 1e-7, first],\n            [c.gamma / (astar * c.sigma * math.sqrt(c.d))] * 2 + [theta.clone()],\n            [1 / astar] * 2 + [theta.clone()]]\n\n\ndef _geom_log(count, h):\n    if count == 0:\n        return torch.full_like(h, -torch.inf)\n    if float(h) == 0:\n        return torch.full_like(h, math.log(count))\n    return torch.log(-torch.expm1(-count * h)) - torch.log(-torch.expm1(-h))\n\n\ndef boundary_log_odds(gaps, m, h, rho, lag, prefix_count):\n    """Exact witness odds: stale a-prefix, correct a, then lag-1 wrong b\'s.\n\n    ``prefix_count=math.inf`` gives the infinite-prefix limit analytically.\n    """\n    if not isinstance(lag, int) or lag < 1 or prefix_count < 0:\n        raise ValueError("lag must be a positive integer and prefix_count nonnegative")\n    result = (-h + _geom_log(prefix_count, h)).expand_as(gaps)\n    if lag >= 2:\n        result = torch.logaddexp(result, -m * (1 - rho) * gaps + h)\n    if lag >= 3:\n        result = torch.logaddexp(result, -m * gaps + (lag - 1) * h + _geom_log(lag - 2, h))\n    return result\n\n\ndef _sequence(keys, values, query, answer, family, pair_id=None):\n    tokens = []\n    for key, value in zip(keys, values):\n        tokens.extend((f"key_{key}", f"value_{value:+d}"))\n    tokens.extend(("?", f"key_{query}"))\n    matching = [i for i, key in enumerate(keys) if key == query]\n    target = matching[-1]\n    return {"family": family, "pair_id": pair_id, "keys": keys, "values": values,\n            "query": query, "answer": answer, "target_lag": len(keys) - target,\n            "tokens": tokens}\n\n\ndef pair_cases(config):\n    """Exactly N(N-1) pair cases; each contains four supervised sequences."""\n    result = []\n    for a in range(config.n):\n        for b in range(config.n):\n            if a == b:\n                continue\n            pair_id = len(result)\n            overwrite = [_sequence([a, a, b, a], [-y, -y, -y, y], a, y, "overwrite", pair_id) for y in (-1, 1)]\n            recall = [_sequence([a] + [b] * (config.R - 1), [y] + [-y] * (config.R - 1),\n                                a, y, "recall", pair_id) for y in (-1, 1)]\n            result.append({"pair_id": pair_id, "query_key": a, "distractor_key": b,\n                           "overwrite": overwrite, "recall": recall})\n    return result\n\n\ndef export_dataset(config, directory):\n    """Export the pair-indexed data and its fully expanded weighted sequences."""\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    cases = pair_cases(config)\n    calibration = [_sequence([a], [y], a, y, "calibration") for a in range(config.n) for y in (-1, 1)]\n    sequences = []\n    for case in cases:\n        for family, weight in (("overwrite", config.wo), ("recall", config.wc)):\n            for row in case[family]:\n                sequences.append({**row, "population_weight": weight / (2 * len(cases))})\n    sequences.extend({**row, "population_weight": config.kcal / len(calibration)} for row in calibration)\n    manifest = {"key_vocabulary_size": config.n, "pair_cases": len(cases), "recall_lag": config.R,\n                "base_recall_sequences": len(cases), "base_overwrite_sequences": len(cases),\n                "paired_sequences_with_complements": 4 * len(cases),\n                "calibration_sequences": len(calibration), "total_expanded_sequences": len(sequences),\n                "mixture_weights": {"calibration": config.kcal, "overwrite": config.wo, "recall": config.wc},\n                "lag_units": "records, newest record at lag 1", "query_is_attention_candidate": False,\n                "dataset_items": "N(N-1) pair cases; each evaluates both families and complements",\n                "files": {"pair_cases": "pairs.jsonl", "expanded_sequences": "dataset.jsonl",\n                          "base_recall_at_R": "recall_at_R.jsonl", "pair_index": "pair_index.csv"}}\n    (directory / "pairs.jsonl").write_text("".join(json.dumps(case) + "\\n" for case in cases))\n    (directory / "dataset.jsonl").write_text("".join(json.dumps(row) + "\\n" for row in sequences))\n    (directory / "recall_at_R.jsonl").write_text("".join(json.dumps(case["recall"][1]) + "\\n" for case in cases))\n    (directory / "pair_index.csv").write_text("pair_id,query_key,distractor_key\\n" + "".join(\n        f\'{case["pair_id"]},{case["query_key"]},{case["distractor_key"]}\\n\' for case in cases))\n    (directory / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n    return manifest\n'


In [ ]:
SOURCES['runner'] = '"""Train the paper\'s restricted answer model; save every finite-run outcome.\n\nThe three acquisition updates use the ordinary loss. Continuation is a\ntractable polynomial schedule, not the existential SGD schedule in the proof.\nAdam keeps its entire moment history and uses the full pair population.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom dataclasses import asdict, replace\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport time\n\nimport numpy as np\nimport torch\n\ntry:\n    from .core import (Config, Model, LogAdam, acquisition_rates, gradient_step,\n                       rates_at, native_gradients, boundary_log_odds, export_dataset)\nexcept ImportError:\n    from core import (Config, Model, LogAdam, acquisition_rates, gradient_step,\n                      rates_at, native_gradients, boundary_log_odds, export_dataset)\n\n\ndef clean(value):\n    if isinstance(value, dict):\n        return {str(k): clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [clean(v) for v in value]\n    if isinstance(value, torch.Tensor):\n        return clean(value.detach().cpu().tolist())\n    if isinstance(value, (float, np.floating)):\n        return float(value) if math.isfinite(value) else None\n    if isinstance(value, np.integer):\n        return int(value)\n    return value\n\n\ndef dump_json(path, value):\n    Path(path).write_text(json.dumps(clean(value), indent=2, allow_nan=False) + "\\n")\n\n\ndef write_csv(path, rows):\n    path = Path(path)\n    if not rows:\n        path.write_text("")\n        return\n    fields = list(dict.fromkeys(key for row in rows for key in row))\n    with path.open("w", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(clean(row) for row in rows)\n\n\ndef fingerprint(params):\n    digest = hashlib.sha256()\n    for value in params:\n        digest.update(value.detach().cpu().contiguous().numpy().tobytes())\n    return digest.hexdigest()\n\n\ndef _validate(c):\n    if c.n < 2 or c.d < 1 or c.R < 2 or c.steps < 3:\n        raise ValueError("Use n>=2, d>=1, integer R>=2, and steps>=3.")\n    if any(type(getattr(c, k)) is not int for k in ("n", "d", "R", "steps", "pair_batch")):\n        raise ValueError("Dimensions, lag, steps and pair batch must be integers.")\n    if not 0 <= c.beta1 < 1 or not c.beta1**2 < c.beta2 < 1:\n        raise ValueError("Adam requires 0<=beta1<1 and beta1**2<beta2<1.")\n    if not 2/3 < c.power <= 1 or c.table_power <= 1:\n        raise ValueError("Use scalar power in (2/3,1] and summable table power>1.")\n    if not .5 < c.kcal < 1 or min(c.wo, c.wc) <= 0 or not math.isclose(c.kcal+c.wo+c.wc, 1):\n        raise ValueError("Positive O/C weights and calibration>.5 must sum to one.")\n    positive = ("sigma", "gamma", "c0", "q0", "u0", "h0", "am", "ag", "ax", "aw",\n                "lr_sgd", "lr_adam", "offset", "table_lr", "eps_decay", "pair_batch")\n    if any(not math.isfinite(getattr(c, k)) or getattr(c, k) <= 0 for k in positive):\n        raise ValueError("Initialization scales and learning-rate settings must be positive and finite.")\n    if c.gamma > 1/16 or c.batch_growth < 0:\n        raise ValueError("Use acquisition gamma<=1/16 and nonnegative batch growth.")\n    if c.ag**2 <= c.c0*c.am**2:\n        raise ValueError("Binder multipliers must satisfy ag**2 > c0*am**2.")\n\n\ndef audit_native(model):\n    """Compare independent native autodiff and analytical signed-log gradients."""\n    actual, native_info = native_gradients(model)\n    signed, info = model.gradients()\n    errors = []\n    for (sign, logabs), reference in zip(signed, actual):\n        reconstructed = sign * logabs.exp()\n        denominator = max(float(reference.abs().max()), 1e-300)\n        errors.append(float((reconstructed-reference).abs().max())/denominator)\n    native_logloss = float(native_info.get("logloss", native_info.get("log_loss")))\n    return dict(relative_max_gradient_error_by_block=errors,\n                absolute_log_loss_error=abs(float(info["logloss"])-native_logloss),\n                passed=max(errors) < 2e-8 and abs(float(info["logloss"])-native_logloss) < 1e-10)\n\n\n@torch.no_grad()\ndef snapshot(model, tags, step, S, initial_tables, examples, elapsed):\n    q,p,u,v,x,w,m,g,h,rho,lrho = [float(v) for v in model.quantities()]\n    gaps = model.gaps()[model.mask]\n    delta = float(gaps.min())\n    logloss = float(model.gradients()[1]["logloss"])\n    _,_,_,so,sc = model.rows(gaps)\n    norms = max(float(model.Q.norm(dim=1).max()), float(model.K.norm(dim=1).max()))\n    movement = max(float((a-b).norm(dim=1).max()) for a,b in zip(model.params()[:2], initial_tables))\n    return dict(**tags, step=step, S=S, examples=examples, elapsed_seconds=elapsed,\n                log_loss=logloss, loss=math.exp(logloss), q=q,p=p,u=u,v=v,x=x,w=w,\n                m=m,g=g,h=h,rho=rho,log_rho=lrho,m_rho=m*rho,\n                delta_min=delta,delta_max=float(gaps.max()),row_norm_max=norms,\n                table_movement=movement, content_horizon=1+m*delta/h,\n                sg_balance=m*delta-(model.cfg.R+1)*h-math.log(m) if m>0 else None,\n                train_overwrite_min_probability=float(torch.sigmoid(w*so).min()),\n                train_recall_min_probability=float(torch.sigmoid(w*sc).min()),\n                calibration_probability=float(torch.sigmoid(model.theta[-1])),\n                certificate_valid=norms<=1 and delta>0 and w>=0 and m>=0,\n                pair_count=model.cfg.n*(model.cfg.n-1), train_R=model.cfg.R)\n\n\n@torch.no_grad()\ndef evaluate(model, tags, step, S, lags, prefixes):\n    """Exhaustive key-pair witness curves, plus an arbitrary-prefix certificate.\n\n    Probe: (a,-1)^P, (a,+1), (b,-1)^(r-1), ?a; complements have\n    exactly the same correct-answer probability. This is not all streams.\n    """\n    _,_,_,_,_,w,m,_,h,rho,_ = model.quantities()\n    gaps = model.gaps()[model.mask]\n    valid = (float(model.Q.norm(dim=1).max())<=1 and float(model.K.norm(dim=1).max())<=1\n             and float(gaps.min())>0 and float(w)>=0 and float(m)>=0)\n    rows = []\n    for lag in lags:\n        logbound = (4*m*rho + torch.logaddexp(-m*gaps.min()+(lag-1)*h, -h)\n                    -torch.log(-torch.expm1(-h)))\n        bound = float(torch.sigmoid(-w*torch.tanh(logbound/2))) if valid else None\n        for prefix in prefixes:\n            logodds = boundary_log_odds(gaps,m,h,rho,lag,prefix)\n            attention = torch.sigmoid(-logodds)\n            probability = torch.sigmoid(-w*torch.tanh(logodds/2))\n            rows.append(dict(**tags,step=step,S=S,lag=lag,prefix=prefix,\n                             mean_probability=float(probability.mean()),min_probability=float(probability.min()),\n                             mean_attention=float(attention.mean()),min_attention=float(attention.min()),\n                             certified_probability_lower_bound=bound,\n                             example_count=2*len(gaps),train_R=model.cfg.R))\n    return rows\n\n\ndef run_suite(cfg, out_dir, seeds=(0,1,2), gate_modes=("learned","retrieval_frozen"),\n              optimizers=("sgd","adam"), eval_lags=tuple(range(1,33)),\n              prefixes=(0,16,64), log_every=1000, progress=True):\n    """Return output path. Never replace an existing experiment.\n\n    `steps` includes three acquisition updates. Scalar clock S starts after\n    them. Sampling RNG and initial tensors are paired across gate variants.\n    Both optimizers start from the Gaussian draw, not an Adam-trained fork.\n    """\n    _validate(cfg)\n    if not seeds or len(set(seeds)) != len(seeds) or any(type(s) is not int or s<0 for s in seeds):\n        raise ValueError("Provide distinct nonnegative integer seeds.")\n    if not gate_modes or not set(gate_modes)<=set(("learned","retrieval_frozen","both_frozen")):\n        raise ValueError("Unknown or empty gate modes.")\n    if not optimizers or not set(optimizers)<=set(("sgd","adam")):\n        raise ValueError("Choose sgd and/or adam.")\n    if len(set(gate_modes))!=len(gate_modes) or len(set(optimizers))!=len(optimizers):\n        raise ValueError("Duplicate branches are not allowed.")\n    if not eval_lags or not prefixes or log_every<1 or any(type(x) is not int or x<1 for x in eval_lags) or any(type(x) is not int or x<0 for x in prefixes):\n        raise ValueError("Use positive integer evaluation lags/log period and nonnegative prefixes.")\n    out = Path(out_dir).resolve()\n    if out.exists() and any(out.iterdir()):\n        raise FileExistsError(f"Run directory is not empty: {out}. Choose a new run tag.")\n    out.mkdir(parents=True,exist_ok=True)\n    (out/"checkpoints").mkdir()\n    source_dir = Path(__file__).resolve().parent\n    source_bytes = {p.name:p.read_bytes() for p in source_dir.glob("*.py")}\n    hashes = {name:hashlib.sha256(data).hexdigest() for name,data in source_bytes.items()}\n    (out/"source").mkdir()\n    for name,data in source_bytes.items():\n        (out/"source"/name).write_bytes(data)\n    dump_json(out/"config.json",dict(config=asdict(cfg),seeds=seeds,gate_modes=gate_modes,\n              optimizers=optimizers,eval_lags=eval_lags,prefixes=prefixes,log_every=log_every,\n              protocol="three ordinary-answer acquisition updates; practical polynomial continuation; no moment reset",\n              numerical_backend="float64 parameters; exact signed-log analytic gradients and bias-corrected Adam buffers",\n              sgd_sampling="full population during acquisition; growing iid ordered-pair batches thereafter",\n              adam_sampling="full ordered-pair population at every update",\n              epsilon="sigma**3 * exp(-eps_decay*(global_step-1)); outside sqrt(v)",\n              source_sha256=hashes,torch_version=torch.__version__,numpy_version=np.__version__,\n              python_version=platform.python_version(),cuda_available=torch.cuda.is_available(),\n              theorem_dimension_at_failure_probability_005=math.ceil(144*math.log(16*cfg.n*(cfg.n-1)/.05)),\n              limits=["Finite schedules do not implement existential SGD continuation envelopes or R>2 basin preparation.",\n                      "Frozen answer theorem is R=2; larger R is an empirical extension.",\n                      "No holdout key pairs: this tests length/lag distribution shift on the same vocabulary.",\n                      "Finite-prefix probe minima are not minima over all possible streams."]))\n    export_dataset(cfg, out/"dataset")\n    histories,probabilities,runs,audits = [],[],[],[]\n    first_hash_by_seed = {}\n    for seed in seeds:\n        for gate_mode in gate_modes:\n            for optimizer in optimizers:\n                c = replace(cfg,seed=seed,gate_mode=gate_mode)\n                model = Model(c)\n                tags = dict(seed=seed,gate_mode=gate_mode,optimizer=optimizer)\n                label = f"seed{seed}_{gate_mode}_{optimizer}"\n                initial_hash = fingerprint(model.params())\n                first_hash_by_seed.setdefault(seed, initial_hash)\n                if initial_hash != first_hash_by_seed[seed]:\n                    raise AssertionError("Paired initial parameters differ across arms.")\n                initial_tables = [p.clone() for p in model.params()[:2]]\n                frozen_initial = model.theta.clone()\n                acquisition = acquisition_rates(c,optimizer)\n                adam = LogAdam(model.params(),c.beta1,c.beta2) if optimizer=="adam" else None\n                rng = np.random.default_rng(seed+104729)\n                start = time.monotonic()\n                S, examples, completed = 0.,0,0\n                status, reason, acquired = "finite_budget_complete", "", None\n                native = audit_native(model)\n                audits.append(dict(**tags,step=0,**native))\n                if not native["passed"]:\n                    raise AssertionError(f"Initial native gradient audit failed: {label}: {native}")\n                def record():\n                    row = snapshot(model,tags,completed,S,initial_tables,examples,time.monotonic()-start)\n                    histories.append(row)\n                    probabilities.extend(evaluate(model,tags,completed,S,eval_lags,prefixes))\n                    return row\n                record()\n                for step in range(1,c.steps+1):\n                    if time.monotonic()-start > c.max_seconds:\n                        status,reason = "time_budget_stop", "per-branch wall-clock limit"\n                        break\n                    if step<=3:\n                        base,rates,counts,batch = 0.,acquisition[step-1],None,c.n*(c.n-1)\n                    else:\n                        tail = step-4\n                        base,rates = rates_at(c,optimizer,tail)\n                        counts = None\n                        batch = c.n*(c.n-1)\n                        if optimizer=="sgd":\n                            batch = math.ceil(c.pair_batch*(1+tail/c.offset)**c.batch_growth)\n                            counts = torch.as_tensor(rng.multinomial(batch,np.full(c.n*(c.n-1),1/(c.n*(c.n-1)))),\n                                                     device=c.device,dtype=torch.float64)\n                    gradients,info = model.gradients(counts)\n                    finite = all(bool(torch.isfinite(s).all()) and bool(torch.isfinite(l[s!=0]).all())\n                                 for s,l in gradients)\n                    if not finite or not math.isfinite(float(info["logloss"])):\n                        status,reason = "numerical_stop","nonfinite signed-log loss or active gradient"\n                        break\n                    if float(info.get("table_log_span",0)) > 650:\n                        status,reason = "numerical_stop","raw-table gradient aggregation dynamic range exceeded 650 log units"\n                        break\n                    previous = [p.clone() for p in model.params()]\n                    if adam:\n                        adam.step(model.params(),gradients,rates,3*math.log(c.sigma)-c.eps_decay*(step-1))\n                    else:\n                        gradient_step(model,gradients,rates)\n                    if not all(bool(torch.isfinite(p).all()) for p in model.params()):\n                        for p,old in zip(model.params(),previous):\n                            p.copy_(old)\n                        status,reason = "numerical_stop","nonfinite parameter update; parameters restored, optimizer state invalid"\n                        break\n                    frozen_indices = getattr(model,"frozen_scalar_indices",())\n                    if frozen_indices and not torch.equal(model.theta[list(frozen_indices)],frozen_initial[list(frozen_indices)]):\n                        raise AssertionError("A frozen coordinate moved.")\n                    S += float(base)\n                    examples += 4*batch+2*c.n\n                    completed = step\n                    if step==3:\n                        acquired = bool((model.gaps()[model.mask]>0).all()) and float(model.theta[-1])>0\n                        native = audit_native(model)\n                        audits.append(dict(**tags,step=3,**native))\n                        if not native["passed"]:\n                            raise AssertionError(f"Acquired native gradient audit failed: {label}: {native}")\n                        if not acquired:\n                            status,reason = "acquisition_failed","three updates did not acquire all positive raw gaps and decoder orientation"\n                    if step==3 or step%log_every==0 or step==c.steps or status!="finite_budget_complete":\n                        row = record()\n                        if progress:\n                            print(f"{label}: {step}/{c.steps}, loss={row[\'loss\']:.3g}, min_gap={row[\'delta_min\']:.4g}, "\n                                  f"horizon={row[\'content_horizon\']:.2f}, S={S:.3g}",flush=True)\n                    if status!="finite_budget_complete":\n                        break\n                if histories[-1]["step"] != completed or any(histories[-1][k]!=v for k,v in tags.items()):\n                    record()\n                run = dict(**tags,completed_steps=completed,requested_steps=c.steps,status=status,reason=reason,\n                           initial_hash=initial_hash,final_hash=fingerprint(model.params()),\n                           acquired_positive_gaps=acquired,train_R=c.R,S=S)\n                runs.append(run)\n                checkpoint = dict(config=asdict(c),model=[p.cpu() for p in model.params()],\n                                  optimizer=None if adam is None else dict(t=adam.t,b1=adam.b1,b2=adam.b2,\n                                       ms=[p.cpu() for p in adam.ms],ml=[p.cpu() for p in adam.ml],vl=[p.cpu() for p in adam.vl]),\n                                  rng_state=rng.bit_generator.state,summary=run,\n                                  optimizer_state_valid="restored" not in reason)\n                torch.save(checkpoint,out/"checkpoints"/(label+".pt"))\n                write_csv(out/"history.csv",histories)\n                write_csv(out/"lag_probabilities.csv",probabilities)\n                write_csv(out/"runs.csv",runs)\n                dump_json(out/"native_gradient_audits.json",audits)\n                if progress:\n                    print(f"{label}: {status}"+(f" ({reason})" if reason else ""),flush=True)\n    return str(out)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out",required=True)\n    parser.add_argument("--n",type=int,default=8)\n    parser.add_argument("--R",type=int,default=2)\n    parser.add_argument("--d",type=int,default=2048)\n    parser.add_argument("--steps",type=int,default=12000)\n    parser.add_argument("--device",default="cpu")\n    parser.add_argument("--seeds",type=int,nargs="+",default=[0,1,2])\n    parser.add_argument("--max-lag",type=int,default=32)\n    parser.add_argument("--log-every",type=int,default=1000)\n    args = parser.parse_args()\n    if args.device=="cpu":\n        torch.set_num_threads(1)\n    cfg = Config(n=args.n,R=args.R,d=args.d,steps=args.steps,device=args.device)\n    path = run_suite(cfg,args.out,seeds=tuple(args.seeds),eval_lags=tuple(range(1,args.max_lag+1)),log_every=args.log_every)\n    try:\n        from .report import make_report\n    except ImportError:\n        from report import make_report\n    make_report(path)\n\n\nif __name__=="__main__":\n    main()\n'


In [ ]:
SOURCES['report'] = '"""Plot the measured restricted-model experiment without importing the trainer.\n\nOnly numpy and matplotlib are required.  CSV files remain the source of truth;\nfigures never extrapolate an optimizer trajectory beyond its saved checkpoints.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\n\n\nCOLORS = {"sgd": "#2166ac", "adam": "#d95f02"}\nGATE_NAMES = {\n    "learned": "Learned retrieval gate",\n    "retrieval_frozen": "Frozen retrieval gate (ALiBi)",\n    "both_frozen": "Both gates frozen (extra control)",\n}\nKEY_FIELDS = ("seed", "gate_mode", "optimizer")\n\n\ndef _number(value, default=float("nan")):\n    try:\n        return float(value)\n    except (TypeError, ValueError):\n        return default\n\n\ndef _read_csv(path):\n    if not path.exists() or not path.stat().st_size:\n        return []\n    with path.open(newline="", encoding="utf-8") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef _key(row):\n    return tuple(str(row.get(field, "")) for field in KEY_FIELDS)\n\n\ndef _group(rows):\n    result = defaultdict(list)\n    for row in rows:\n        result[_key(row)].append(row)\n    return result\n\n\ndef _latest(rows):\n    """Select the last evaluated checkpoint separately for every branch."""\n    result = []\n    for branch in _group(rows).values():\n        step = max((_number(row.get("step"), -1) for row in branch), default=-1)\n        result.extend(row for row in branch if _number(row.get("step"), -1) == step)\n    return result\n\n\ndef _mean(values):\n    values = [v for v in values if math.isfinite(v)]\n    return float(np.mean(values)) if values else float("nan")\n\n\ndef _minimum(values):\n    values = [v for v in values if math.isfinite(v)]\n    return min(values) if values else float("nan")\n\n\ndef _per_seed_lag(rows, metric="min_probability"):\n    # Each CSV row already aggregates ordered key pairs for one prefix. Taking\n    # a second minimum therefore gives a minimum over the finite test panel.\n    groups = defaultdict(list)\n    for row in rows:\n        groups[(_key(row), _number(row.get("lag")))].append(_number(row.get(metric)))\n    result = defaultdict(dict)\n    for (branch, lag), values in groups.items():\n        if math.isfinite(lag):\n            result[branch][lag] = _minimum(values)\n    return result\n\n\ndef _per_seed_mean_lag(rows):\n    groups = defaultdict(list)\n    for row in rows:\n        value = _number(row.get("mean_probability"))\n        weight = _number(row.get("example_count"), 1)\n        lag = _number(row.get("lag"))\n        if math.isfinite(value) and math.isfinite(weight) and weight > 0 and math.isfinite(lag):\n            groups[(_key(row), lag)].append((value, weight))\n    result = defaultdict(dict)\n    for (key, lag), values in groups.items():\n        result[key][lag] = sum(v * w for v, w in values) / sum(w for _, w in values)\n    return result\n\n\ndef _format(value, digits=4):\n    value = _number(value)\n    return f"{value:.{digits}g}" if math.isfinite(value) else "unavailable"\n\n\ndef _save(fig, path, pdf=False):\n    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")\n    if pdf:\n        fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")\n    plt.close(fig)\n\n\ndef _empty(ax, message="No saved observations"):\n    ax.text(.5, .5, message, ha="center", va="center", transform=ax.transAxes, color="0.4")\n\n\ndef _finish_axes(axes):\n    for ax in np.asarray(axes).ravel():\n        ax.grid(alpha=.17)\n        ax.spines[["top", "right"]].set_visible(False)\n        handles, labels = ax.get_legend_handles_labels()\n        if handles:\n            unique = dict(zip(labels, handles))\n            ax.legend(unique.values(), unique.keys(), fontsize=8, frameon=False,\n                      ncol=2 if len(unique) > 6 else 1)\n\n\ndef _lag_plot(final_rows, gates, R, path):\n    fig, axes = plt.subplots(1, len(gates), figsize=(6.3 * len(gates), 4.6), squeeze=False)\n    series = _per_seed_lag(final_rows)\n    mean_series = _per_seed_mean_lag(final_rows)\n    for ax, gate in zip(axes.ravel(), gates):\n        has_data = False\n        for opt, color in COLORS.items():\n            branches = [vals for key, vals in series.items() if key[1:] == (gate, opt)]\n            if not branches:\n                continue\n            has_data = True\n            for values in branches:\n                xs = sorted(values)\n                ax.plot(xs, [values[x] for x in xs], color=color, alpha=.18, lw=.9)\n            xs = sorted({lag for values in branches for lag in values})\n            # Unequal finite run budgets remain explicit in report.md. This\n            # envelope is a seed range, not a confidence interval.\n            batches = [[v[x] for v in branches if x in v and math.isfinite(v[x])] for x in xs]\n            means = [_mean(batch) for batch in batches]\n            lower = [_minimum(batch) for batch in batches]\n            upper = [max(batch) if batch else float("nan") for batch in batches]\n            ax.fill_between(xs, lower, upper, color=color, alpha=.12)\n            ax.plot(xs, means, color=color, lw=2.2, label=f"{opt.upper()}: seed mean of minima")\n            mean_branches = [vals for key, vals in mean_series.items() if key[1:] == (gate, opt)]\n            if mean_branches:\n                ax.plot(xs, [_mean([vals[x] for vals in mean_branches if x in vals]) for x in xs],\n                        color=color, ls="--", lw=1.3, label=f"{opt.upper()}: mean probability")\n        for threshold in (.5, .9):\n            ax.axhline(threshold, color=".55", linestyle=":", lw=.8)\n            ax.text(.985, threshold + .01, f"{threshold:g}", transform=ax.get_yaxis_transform(),\n                    ha="right", fontsize=8, color=".4")\n        if R is not None:\n            ax.axvline(R, color=".3", linestyle="--", lw=1, label=f"Training lag R = {R}")\n            if gate == "learned" and R >= 2:\n                ax.axvline(R + 2, color="#6a51a3", linestyle=":", lw=1.4,\n                           label=("SGD horizon 4 (asymptotic theorem)" if R == 2\n                                  else "SGD horizon R + 2 (conditional asymptotic)"))\n        if not has_data:\n            _empty(ax)\n        ax.set(title=GATE_NAMES.get(gate, gate), xlabel="Target lag r (records)",\n               ylabel="Correct-answer probability", ylim=(-.025, 1.045))\n    fig.suptitle("Generalization at the latest saved checkpoint of each run", fontsize=14)\n    fig.text(.5, .015, "Solid: seed mean of finite-panel minima. Dashed: mean over tested examples, then seeds. "\n             "Band: range of seed minima. Finite runs do not establish asymptotic limits.", ha="center", fontsize=8)\n    _finish_axes(axes)\n    fig.tight_layout(rect=(0, .06, 1, .94))\n    _save(fig, path, pdf=True)\n\n\ndef _training_plot(rows, history, gates, R, path):\n    fig, axes = plt.subplots(len(gates), 2, figsize=(12.7, 3.65 * len(gates)), squeeze=False)\n    history_lookup = {(_key(row), _number(row.get("step"))): row for row in history}\n    chosen_lags = [R, R + 2, R + 3, 2 * R + 4] if R is not None else []\n    if not chosen_lags:\n        chosen_lags = sorted({_number(row.get("lag")) for row in rows if math.isfinite(_number(row.get("lag")))})[:4]\n    chosen_lags = list(dict.fromkeys(chosen_lags))\n    lag_colors = ["#1b9e77", "#7570b3", "#d95f02", "#e7298a"]\n    aggregated = defaultdict(list)\n    for row in rows:\n        aggregated[(_key(row), _number(row.get("step")), _number(row.get("lag")))].append(row)\n    curves = defaultdict(list)\n    for (branch, step, lag), batch in aggregated.items():\n        source = history_lookup.get((branch, step), {})\n        S = _number(batch[0].get("S"))\n        if not math.isfinite(S):\n            S = _number(source.get("S"))\n        value = _minimum([_number(row.get("min_probability")) for row in batch])\n        if math.isfinite(S) and math.isfinite(value):\n            curves[(branch, lag)].append((S, step, value))\n    for row_index, gate in enumerate(gates):\n        for col_index, opt in enumerate(COLORS):\n            ax = axes[row_index, col_index]\n            has_data = False\n            seed_styles = {}\n            for lag, color in zip(chosen_lags, lag_colors):\n                branches = [(key, values) for (key, r), values in curves.items() if key[1:] == (gate, opt) and r == lag]\n                for branch_index, (key, values) in enumerate(sorted(branches)):\n                    values = sorted(values, key=lambda point: point[1])\n                    linestyle = ("-", "--", "-.", ":")[branch_index % 4]\n                    seed_styles[key[0]] = linestyle\n                    ax.plot([p[0] for p in values], [p[2] for p in values], color=color,\n                            lw=1.5, alpha=.8, ls=linestyle,\n                            label=f"r = {lag:g}" if branch_index == 0 else "_nolegend_")\n                    has_data = True\n            for seed, linestyle in sorted(seed_styles.items()):\n                ax.plot([], [], color=".4", ls=linestyle, lw=1.2, label=f"seed {seed}")\n            ax.axhline(.5, color=".55", ls=":", lw=.8)\n            ax.axhline(.9, color=".55", ls=":", lw=.8)\n            ax.set_xscale("symlog", linthresh=1)\n            ax.set(title=f"{GATE_NAMES.get(gate, gate)} · {opt.upper()}",\n                   xlabel="Continuation learning-rate sum S (own run schedule)",\n                   ylabel="Minimum correct-answer probability", ylim=(-.025, 1.045))\n            if not has_data:\n                _empty(ax, "No saved probabilities at the selected lags")\n    fig.suptitle("Probability during training at selected test lags", fontsize=14)\n    fig.text(.5, .012, "Each seed keeps its own learning-rate sum; curves are not aligned or averaged across different S values.",\n             ha="center", fontsize=9)\n    _finish_axes(axes)\n    fig.tight_layout(rect=(0, .045, 1, .95))\n    _save(fig, path)\n\n\ndef _diagnostic_plot(history, final_rows, gates, R, path):\n    fig, axes = plt.subplots(len(gates), 4, figsize=(19, 3.9 * len(gates)), squeeze=False)\n    specifications = [\n        ("log_loss", "Log ordinary answer loss", "ln(loss)"),\n        ("content_horizon", "Content horizon", "1 + m δ_min / h"),\n        ("m_rho", "Binding-contamination scale", "m ρ"),\n    ]\n    for gate_index, gate in enumerate(gates):\n        for metric_index, (metric, title, ylabel) in enumerate(specifications):\n            ax = axes[gate_index, metric_index]\n            has_data = False\n            for key, branch in sorted(_group(history).items()):\n                if key[1] != gate:\n                    continue\n                points = []\n                for row in sorted(branch, key=lambda row: _number(row.get("step"), -1)):\n                    x, y = _number(row.get("S")), _number(row.get(metric))\n                    if metric == "log_loss" and not math.isfinite(y):\n                        loss = _number(row.get("loss"))\n                        y = math.log(loss) if loss > 0 else float("nan")\n                    if math.isfinite(x) and math.isfinite(y):\n                        points.append((x, y))\n                if points:\n                    ax.plot(*zip(*points), color=COLORS.get(key[2], ".3"), lw=1.5,\n                            alpha=.8, label=f"{key[2].upper()}, seed {key[0]}")\n                    has_data = True\n            ax.set_xscale("symlog", linthresh=1)\n            if metric == "m_rho":\n                ax.set_yscale("symlog", linthresh=1e-7)\n            if metric == "content_horizon" and R is not None:\n                ax.axhline(R, color=".5", ls="--", lw=.8, label="Training lag R")\n                if gate == "learned" and R >= 2:\n                    ax.axhline(R + 2, color="#6a51a3", ls=":", lw=1,\n                               label=("SGD asymptote 4 (R = 2)" if R == 2\n                                      else "SGD R + 2 (conditional, R > 2)"))\n            ax.set(title=f"{GATE_NAMES.get(gate, gate)}\\n{title}", xlabel="Continuation sum S", ylabel=ylabel)\n            if not has_data:\n                _empty(ax)\n        ax = axes[gate_index, 3]\n        for metric, label, linestyle in [("min_probability", "answer", "-"), ("min_attention", "attention", "--")]:\n            series = _per_seed_lag(final_rows, metric)\n            for opt, color in COLORS.items():\n                branches = [vals for key, vals in series.items() if key[1:] == (gate, opt)]\n                xs = sorted({lag for values in branches for lag in values})\n                if xs:\n                    ax.plot(xs, [_mean([v[x] for v in branches if x in v]) for x in xs],\n                            color=color, ls=linestyle, lw=1.8, label=f"{opt.upper()} {label}")\n        ax.axhline(.5, color=".55", ls=":", lw=.8)\n        ax.set(title="Latest checkpoint: attention vs answer", xlabel="Target lag r (records)",\n               ylabel="Seed mean of finite-panel minima", ylim=(-.025, 1.045))\n        if not final_rows:\n            _empty(ax)\n    fig.suptitle("Mechanism diagnostics from saved checkpoints", fontsize=15)\n    fig.text(.5, .012, "A large content horizon is a diagnostic, not a probability certificate; "\n             "answer confidence also depends on binding, old prefixes, and output scale.", ha="center", fontsize=9)\n    _finish_axes(axes)\n    fig.tight_layout(rect=(0, .045, 1, .95))\n    _save(fig, path)\n\n\ndef _markdown(config_data, history, lag_rows, runs, R):\n    config = config_data.get("config", config_data)\n    final_rows = _latest(lag_rows)\n    final_history = {_key(row): row for row in _latest(history)}\n    settings = []\n    for name in ("n", "d", "R", "steps"):\n        if name in config:\n            settings.append(f"{name}={config[name]}")\n    if "R" not in config and R is not None:\n        settings.append(f"R={R}")\n    lines = ["# Restricted-model empirical report", "",\n             "These results describe the saved finite training runs. They can test consistency with the theory; "\n             "they do not prove the asymptotic optimizer claims.", ""]\n    if settings:\n        lines.extend(["Settings: " + ", ".join(settings) + ". See `config.json` for initialization, optimizer, and evaluation settings.", ""])\n    prefixes = sorted({_number(row.get("prefix")) for row in lag_rows if math.isfinite(_number(row.get("prefix")))})\n    lags = sorted({_number(row.get("lag")) for row in lag_rows if math.isfinite(_number(row.get("lag")))})\n    lines.extend([\n        "- Checkpoint step 0 is the paired Gaussian initialization; step 3 ends optimizer-specific acquisition. "\n        "Continuation uses each optimizer\'s own learning-rate sum S.",\n        "- The frozen-retrieval comparison freezes only the retrieval forgetting parameter h. "\n        "The binding gate remains trainable. Any `both_frozen` arm is a separate extra control.",\n        "- The SGD continuation schedule is a practical experiment schedule, not the literal conservative theorem schedule.",\n        "- Frozen-gate arbitrary-prefix success requires the theorem\'s conditions, including h₀ > log(2). "\n        "Positive matching gaps, suitable row bounds, controlled binding contamination, and positive output scale must also hold.",\n        "- Learned-gate SGD\'s asymptotic horizon is 4 in the main R = 2 theorem. "\n        "The R + 2 reference for R > 2 is conditional on the generalized theorem\'s hypotheses. "\n        "At the boundary, output probability must be checked separately from target attention.",\n        "- The fully proved frozen-retrieval answer result uses R=2. Frozen R>2 and learned-Adam R>2 "\n        "runs are empirical extension tests here; the notebook also omits the additional R>2 SGD basin preparation.",\n        "- Test prefix lengths: " + (", ".join(f"{p:g}" for p in prefixes) or "none saved") + ". "\n        "Test lags: " + (", ".join(f"{r:g}" for r in lags) or "none saved") + ".", "",\n        "## Run completion and latest observations", "",\n        "Plots use the latest saved evaluation separately for each branch. A stopped run may therefore have an earlier "\n        "checkpoint than a completed run. Seed envelopes are observed ranges, not confidence intervals.", "",\n        "| Seed | Gate | Optimizer | Completed / requested | Status | Evaluation step | Reason |",\n        "| --- | --- | --- | --- | --- | --- | --- |",\n    ])\n    final_steps = {_key(row): _format(row.get("step"), 8) for row in final_rows}\n    branch_runs = list(runs)\n    known = {_key(row) for row in branch_runs}\n    for key, row in final_history.items():\n        if key not in known:\n            branch_runs.append(dict(row, completed_steps=row.get("step", ""), requested_steps="unavailable"))\n    for row in sorted(branch_runs, key=_key):\n        key = _key(row)\n        reason = str(row.get("reason", "")).replace("|", "/").replace("\\n", " ") or "—"\n        lines.append(f"| {key[0]} | {key[1]} | {key[2]} | {row.get(\'completed_steps\', \'unavailable\')} / "\n                     f"{row.get(\'requested_steps\', \'unavailable\')} | {row.get(\'status\', \'unavailable\')} | "\n                     f"{final_steps.get(key, \'none\')} | {reason} |")\n    if not branch_runs:\n        lines.append("| — | — | — | — | No saved runs | — | — |")\n    lines.extend(["", "## Generalization at selected lags", "",\n                  "For each seed, the observed minimum is over all tested ordered key pairs and prefix lengths. "\n                  "The table gives the mean and worst of those per-seed minima. It does not cover untested prefixes. "\n                  "An analytic certificate, when present, is a separate conditional arbitrary-prefix lower bound; "\n                  "blank or unavailable certificates are not empirical failures.", "",\n                  "| Gate | Optimizer | Lag | Seeds | Mean probability | Mean of observed minima | Worst observed minimum | Analytic lower bound (worst seed) |",\n                  "| --- | --- | --- | --- | --- | --- | --- | --- |"])\n    selected = set([R, R + 2, R + 3, 2 * R + 4]) if R is not None else set(lags[:4])\n    series = _per_seed_lag(final_rows)\n    mean_series = _per_seed_mean_lag(final_rows)\n    certificates = _per_seed_lag(final_rows, "certified_probability_lower_bound")\n    groups = defaultdict(list)\n    for key, values in series.items():\n        for lag, value in values.items():\n            if lag in selected:\n                groups[(key[1], key[2], lag)].append((key, value))\n    for (gate, opt, lag), observations in sorted(groups.items()):\n        values = [value for _, value in observations]\n        bounds = [certificates.get(key, {}).get(lag, float("nan")) for key, _ in observations]\n        mean_probability = _mean([mean_series.get(key, {}).get(lag, float("nan")) for key, _ in observations])\n        # Never label a partial collection as a certificate across every seed.\n        bound = min(bounds) if bounds and all(math.isfinite(v) for v in bounds) else float("nan")\n        lines.append(f"| {gate} | {opt} | {lag:g} | {len(values)} | {_format(mean_probability)} | {_format(_mean(values))} | "\n                     f"{_format(_minimum(values))} | {_format(bound)} |")\n    if not groups:\n        lines.append("| — | — | — | 0 | unavailable | unavailable | unavailable | unavailable |")\n    lines.extend(["", "## Final mechanism diagnostics", "",\n                  "| Seed | Gate | Optimizer | Step | ln(loss) | δ_min | Max row norm | h | mρ | Content horizon | SGD balance |",\n                  "| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |"])\n    for key, row in sorted(final_history.items()):\n        metrics = [row.get(name) for name in ("step", "log_loss", "delta_min", "row_norm_max", "h", "m_rho", "content_horizon", "sg_balance")]\n        lines.append("| " + " | ".join([*key, *[_format(v) for v in metrics]]) + " |")\n    if not final_history:\n        lines.append("| — | — | — | — | unavailable | unavailable | unavailable | unavailable | unavailable | unavailable | unavailable |")\n    lines.extend(["", "The content horizon is 1 + m δ_min / h; the SGD balance is m δ_min − (R + 1)h − log(m). "\n                  "These are mechanism diagnostics, not substitutes for the actual answer probabilities. "\n                  "`theory_diagnostics.png` displays target attention separately from correct-answer probability, "\n                  "since the two can behave differently.", "",\n                  "## Saved figures", "",\n                  "- `generalization_by_lag.png` and `.pdf`: probability versus lag at each branch\'s latest evaluation.",\n                  "- `probability_by_training.png`: selected-lag probabilities along each saved training trajectory.",\n                  "- `theory_diagnostics.png`: log loss, content horizon, binding contamination, and attention versus answer probability.",\n                  "", "The CSV files contain all measurements and stopped-run statuses. Unobserved checkpoints are never filled in.", ""])\n    return "\\n".join(lines)\n\n\ndef make_report(out_dir):\n    """Read an experiment directory and return the generated artifact paths.\n\n    Empty or stopped experiments still produce explicit, readable placeholder\n    figures and a status report. Nothing is imputed as a successful outcome.\n    """\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    history = _read_csv(out_dir / "history.csv")\n    lag_rows = _read_csv(out_dir / "lag_probabilities.csv")\n    runs = _read_csv(out_dir / "runs.csv")\n    config_path = out_dir / "config.json"\n    config_data = json.loads(config_path.read_text(encoding="utf-8")) if config_path.exists() else {}\n    config = config_data.get("config", config_data)\n    R = _number(config.get("R", config.get("train_R")))\n    if not math.isfinite(R):\n        R = next((_number(row.get("train_R")) for row in history + runs if math.isfinite(_number(row.get("train_R")))), float("nan"))\n    R = int(R) if math.isfinite(R) else None\n    observed_gates = {str(row.get("gate_mode")) for row in history + lag_rows + runs if row.get("gate_mode")}\n    observed_gates.update(config_data.get("gate_modes", []))\n    gates = [gate for gate in GATE_NAMES if gate in observed_gates]\n    gates += sorted(observed_gates - set(gates))\n    gates = gates or ["learned", "retrieval_frozen"]\n    paths = {\n        "generalization_by_lag": str(out_dir / "generalization_by_lag.png"),\n        "generalization_by_lag_pdf": str(out_dir / "generalization_by_lag.pdf"),\n        "probability_by_training": str(out_dir / "probability_by_training.png"),\n        "theory_diagnostics": str(out_dir / "theory_diagnostics.png"),\n        "report": str(out_dir / "report.md"),\n    }\n    with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 11}):\n        _lag_plot(_latest(lag_rows), gates, R, Path(paths["generalization_by_lag"]))\n        _training_plot(lag_rows, history, gates, R, Path(paths["probability_by_training"]))\n        _diagnostic_plot(history, _latest(lag_rows), gates, R, Path(paths["theory_diagnostics"]))\n    Path(paths["report"]).write_text(_markdown(config_data, history, lag_rows, runs, R), encoding="utf-8")\n    return paths\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("out_dir", type=Path, help="Directory containing experiment CSV files")\n    args = parser.parse_args()\n    for name, path in make_report(args.out_dir).items():\n        print(f"{name}: {path}")\n'


In [ ]:
SOURCES['asymptotics'] = '"""Uncapped, arbitrary-context diagnostics for the paper\'s answer error.\n\nE_t(r) is the supremum of 1-P(correct) over *all finite streams* whose latest\nqueried record has lag at most r.  An infinite stale-prefix witness gives a\nlower bound on this supremum (by taking limits of finite witnesses).  The\nmanuscript\'s attention-odds transfer inequality gives an upper bound whenever\nits row-norm, positive-gap, and sign hypotheses hold. Neither bound is called\nan exact computation of E_t(r).\n\nNo sequence of length r is materialized. Errors are evaluated as\nsigmoid(-correct_logit), and log errors as logsigmoid(-correct_logit), never\nby subtracting a saturated probability from one.\n"""\nfrom dataclasses import dataclass\nimport math\n\nimport torch\nfrom torch.nn import functional as F\n\ntry:\n    from .core import _geom_log\nexcept ImportError:\n    from core import _geom_log\n\n\n@dataclass\nclass ErrorSnapshot:\n    gaps: torch.Tensor\n    m: torch.Tensor\n    h: torch.Tensor\n    rho: torch.Tensor\n    w: torch.Tensor\n    delta_min: float\n    max_row_norm: float\n    bound_valid: bool\n    bound_invalid_reason: str\n\n\n@torch.no_grad()\ndef snapshot(model):\n    """Copy compact quantities once, avoiding repeated device synchronizations."""\n    quantities = torch.stack(list(model.quantities()[5:10])).detach().cpu()\n    w, m, _, h, rho = quantities.unbind()\n    gaps = model.gaps()[model.mask].detach().cpu()\n    row_norm = float(torch.maximum(model.Q.norm(dim=1).max(), model.K.norm(dim=1).max()))\n    delta = float(gaps.min())\n    reasons = []\n    if not bool(torch.isfinite(quantities).all()) or not bool(torch.isfinite(gaps).all()) or not math.isfinite(row_norm):\n        reasons.append("nonfinite_model")\n    if row_norm > 1:\n        reasons.append("row_norm_exceeds_one")\n    if delta <= 0:\n        reasons.append("nonpositive_matching_gap")\n    if float(m) < 0:\n        reasons.append("negative_matching_scale")\n    if float(w) < 0:\n        reasons.append("negative_decoder")\n    if float(h) <= 0:\n        reasons.append("nonpositive_forgetting")\n    if not 0 <= float(rho) <= 1:\n        reasons.append("invalid_binding_probability")\n    return ErrorSnapshot(gaps, m, h, rho, w, delta, row_norm, not reasons, ";".join(reasons))\n\n\ndef _positive_integer(value, name):\n    if isinstance(value, bool) or not isinstance(value, int) or value < 1:\n        raise ValueError(f"{name} must be a positive integer")\n\n\ndef _error_from_log_odds(log_odds, w):\n    # z=(1-A)/(1+A)=-tanh(log(A)/2), so error=sigmoid(w*tanh(log(A)/2)).\n    return F.logsigmoid(w * torch.tanh(log_odds / 2))\n\n\ndef _float_integer(value):\n    # Torch first interprets Python ints as int64. Convert explicitly so lags\n    # beyond 2**63 still work without changing the recorded Python integer.\n    try:\n        return float(value)\n    except OverflowError:\n        return math.inf\n\n\ndef _witness_log_odds(state, lag):\n    result = (-state.h + _geom_log(math.inf, state.h)).expand_as(state.gaps)\n    if lag >= 2:\n        result = torch.logaddexp(result, -state.m * (1 - state.rho) * state.gaps + state.h)\n    if lag >= 3:\n        result = torch.logaddexp(result, -state.m * state.gaps + _float_integer(lag - 1) * state.h\n                                + _geom_log(_float_integer(lag - 2), state.h))\n    return result\n\n\ndef _log_uniform_odds(state, lag):\n    log_tail = _geom_log(math.inf, state.h)\n    competing = -state.m * state.delta_min + _float_integer(lag - 1) * state.h\n    return 4 * state.m * state.rho + torch.logaddexp(competing, -state.h) + log_tail\n\n\n@torch.no_grad()\ndef error_bounds(state, lag):\n    """Lower witness and conditional upper bound on E_t(lag).\n\n    The witness is (a,-1)^k,(a,+1),(b,-1)^(lag-1), maximized over all\n    a!=b, as k tends to infinity. For w>=0 its error increases with lag:\n    extending the suffix adds positive non-target attention odds while leaving\n    the target-relative old tail unchanged. Thus it is a valid lower bound for\n    the nested lag<=r sets. Positive matching gaps are needed for the stated\n    uniform upper certificate, not for this witness construction.\n    """\n    _positive_integer(lag, "lag")\n    odds = _witness_log_odds(state, lag)\n    log_lower = float(_error_from_log_odds(odds, state.w).max())\n    log_upper = float(_error_from_log_odds(_log_uniform_odds(state, lag), state.w)) if state.bound_valid else None\n    return {\n        "lag": lag,\n        "lower_error": math.exp(log_lower),\n        "upper_error": math.exp(log_upper) if log_upper is not None else None,\n        "log_lower_error": log_lower,\n        "log_upper_error": log_upper,\n        "bound_valid": state.bound_valid,\n        "bound_invalid_reason": state.bound_invalid_reason,\n        "lower_bound_method": "infinite_stale_prefix_max_all_pairs",\n        "upper_bound_method": "uniform_arbitrary_stream_transfer" if state.bound_valid else "",\n    }\n\n\ndef scalar_snapshot(delta_min, m, h, rho, w, row_norm_max=1.):\n    """Reconstruct the diagnostics state from a scalar checkpoint history.\n\n    For m,w>=0, each wrong-key term decreases with the raw gap, so the\n    minimum-gap pair attains the all-pairs maximum witness error. Other signs\n    are rejected because delta_min alone no longer identifies that maximum.\n    """\n    if m < 0 or w < 0:\n        raise ValueError("scalar_snapshot requires m,w>=0; use the full snapshot for other signs")\n    values = (delta_min, m, h, rho, w, row_norm_max)\n    reasons = []\n    if not all(math.isfinite(value) for value in values):\n        reasons.append("nonfinite_model")\n    if delta_min <= 0:\n        reasons.append("nonpositive_matching_gap")\n    if row_norm_max > 1:\n        reasons.append("row_norm_exceeds_one")\n    if h <= 0:\n        reasons.append("nonpositive_forgetting")\n    if not 0 <= rho <= 1:\n        reasons.append("invalid_binding_probability")\n    tensors = torch.tensor([m, h, rho, w], dtype=torch.float64)\n    return ErrorSnapshot(torch.tensor([delta_min], dtype=torch.float64), *tensors.unbind(),\n                         delta_min, row_norm_max, not reasons, ";".join(reasons))\n\n\ndef scalar_error_bounds(delta_min, m, h, rho, w, lag, row_norm_max=1.):\n    """Compute E_t(lag) bounds directly from saved scalar diagnostics."""\n    return error_bounds(scalar_snapshot(delta_min, m, h, rho, w, row_norm_max), lag)\n\n\ndef certified_radius(state, error_target):\n    """Largest integer radius certified by the analytic upper bound.\n\n    Zero means the bound cannot certify even lag one; None means its\n    hypotheses fail. This is *not* the exact maximal generalizing radius.\n    The analytic inverse is checked against adjacent integers to make the\n    floor decision robust near a floating-point boundary.\n    """\n    if not math.isfinite(error_target) or not 0 < error_target < .5:\n        raise ValueError("error_target must lie strictly between zero and 0.5")\n    result = {"error_target": error_target, "certified_radius": None,\n              "bound_valid": state.bound_valid, "bound_invalid_reason": state.bound_invalid_reason,\n              "radius_definition": "max_radius_certified_by_uniform_upper_bound"}\n    if not state.bound_valid:\n        return result\n    w, m, h, rho = (float(value) for value in (state.w, state.m, state.h, state.rho))\n    required_margin = math.log1p(-error_target) - math.log(error_target)\n    if w <= required_margin:\n        result["certified_radius"] = 0\n        return result\n    required_z = required_margin / w\n    allowed_log_odds = math.log1p(-required_z) - math.log1p(required_z)\n    log_denominator = math.log(-math.expm1(-h))\n    remaining = allowed_log_odds - 4 * m * rho + log_denominator\n    if remaining <= -h:\n        result["certified_radius"] = 0\n        return result\n    # log(exp(remaining)-exp(-h)); expm1 is accurate near equality.\n    log_recent_allowance = remaining + math.log(-math.expm1(-h - remaining))\n    radius_real = 1 + (m * state.delta_min + log_recent_allowance) / h\n    if not math.isfinite(radius_real):\n        result["bound_valid"] = False\n        result["bound_invalid_reason"] = "radius_inverse_exceeds_float64_range"\n        return result\n    radius = max(0, math.floor(radius_real))\n    target_log = math.log(error_target)\n    def passes(r):\n        return float(_error_from_log_odds(_log_uniform_odds(state, r), state.w)) <= target_log\n    # In ordinary ranges these loops execute zero or one times. Above 2**53,\n    # adjacent integers cannot be distinguished in float64; keep a conservative\n    # representable radius rather than spending time walking the plateau.\n    if radius > 2 ** 53:\n        radius = max(0, math.floor(math.nextafter(radius_real, -math.inf)))\n        result["radius_rounding"] = "conservative_float64_spacing"\n    else:\n        while radius > 0 and not passes(radius):\n            radius -= 1\n        while radius < 2 ** 53 and passes(radius + 1):\n            radius += 1\n        result["radius_rounding"] = "adjacent_integer_verified"\n    result["certified_radius"] = radius\n    return result\n\n\ndef _moving_lag(state, theta, gap, learned):\n    if not math.isfinite(gap) or gap <= 0 or float(state.h) <= 0 or not math.isfinite(float(state.m / state.h)):\n        return 1\n    # Learned reference: floor(theta*m*delta/h-1). Frozen reference:\n    # floor(1+theta*m*delta/h). Clamping makes early checkpoints evaluable;\n    # a flat/clamped radius is not evidence for a diverging-radius theorem.\n    return max(1, math.floor(theta * float(state.m / state.h) * gap + (-1 if learned else 1)))\n\n\n@torch.no_grad()\ndef evaluate_asymptotics(model, tags, step, S, reference_gap=None,\n                         fixed_lags=(1, 2, 4, 8, 16, 32, 64, 128, 256, 512),\n                         theta_values=(.25, .5, .75), clock_coefficients=(.001, .003, .01),\n                         error_targets=(.1, .01, .001)):\n    """Return flat error-trajectory rows and certified-radius rows.\n\n    ``adaptive_theta`` uses the current minimum gap. ``reference_theta``\n    uses a gap recorded just after acquisition; its retention is logged, never\n    assumed. Neither is an exogenous schedule. ``clock`` is predeclared as\n    floor(1+c*S**a), with a=1 learned / a=2 frozen. The learning-rate clock\n    S is fixed by the training schedule, and c is never fitted to test error.\n    No evaluation lag cap is applied to any schedule.\n    """\n    return evaluate_snapshot(snapshot(model), model.cfg.R, model.cfg.gate_mode, tags, step, S,\n                             reference_gap, fixed_lags, theta_values, clock_coefficients, error_targets)\n\n\n@torch.no_grad()\ndef evaluate_snapshot(state, train_R, gate_mode, tags, step, S, reference_gap=None,\n                      fixed_lags=(1, 2, 4, 8, 16, 32, 64, 128, 256, 512),\n                      theta_values=(.25, .5, .75), clock_coefficients=(.001, .003, .01),\n                      error_targets=(.1, .01, .001)):\n    """The same evaluator for current models and saved scalar histories.\n\n    Scalar histories must include their actual minimum matching gap, scale,\n    gate, binding probability, decoder, and maximum raw row norm. No state is\n    interpolated or extrapolated between saved checkpoints.\n    """\n    _positive_integer(train_R, "train_R")\n    if gate_mode not in ("learned", "retrieval_frozen", "both_frozen"):\n        raise ValueError("Unknown gate mode")\n    if not math.isfinite(S) or S < 0:\n        raise ValueError("S must be nonnegative and finite")\n    for theta in theta_values:\n        if not math.isfinite(theta) or not 0 < theta < 1:\n            raise ValueError("theta_values must be in (0,1)")\n    for coefficient in clock_coefficients:\n        if not math.isfinite(coefficient) or coefficient <= 0:\n            raise ValueError("clock_coefficients must be positive and finite")\n    learned = gate_mode == "learned"\n    power = 1 if learned else 2\n    common = {**tags, "step": step, "S": S, "delta_min": state.delta_min,\n              "max_row_norm": state.max_row_norm, "m": float(state.m), "h": float(state.h),\n              "w": float(state.w), "m_rho": float(state.m * state.rho),\n              "reference_gap": reference_gap,\n              "reference_gap_retained": state.delta_min >= reference_gap if reference_gap is not None else None}\n    probes = []\n    lags = set(fixed_lags) | {train_R + offset for offset in range(4)}\n    for lag in sorted(lags):\n        _positive_integer(lag, "fixed_lag")\n        probes.append(("fixed", lag, None, None, None))\n    for theta in theta_values:\n        probes.append(("adaptive_theta", _moving_lag(state, theta, state.delta_min, learned), theta, None, None))\n        if reference_gap is not None and math.isfinite(reference_gap) and reference_gap > 0:\n            probes.append(("reference_theta", _moving_lag(state, theta, reference_gap, learned), theta, None, None))\n    for coefficient in clock_coefficients:\n        lag = max(1, math.floor(1 + coefficient * S ** power))\n        probes.append(("clock", lag, None, power, coefficient))\n    cache = {}\n    error_rows = []\n    for probe, lag, theta, clock_power, coefficient in probes:\n        if lag not in cache:\n            cache[lag] = error_bounds(state, lag)\n        error_rows.append({**common, "probe": probe, "theta": theta, "clock_power": clock_power,\n                           "coefficient": coefficient, **cache[lag]})\n    radius_rows = [{**common, **certified_radius(state, target)} for target in error_targets]\n    return error_rows, radius_rows\n'


In [ ]:
SOURCES['a100_runner'] = '"""Long-run restricted experiments with exact optimizer-state continuation.\n\nCheckpoints commit validated states by writing a temporary sibling and replacing\nthe destination. A process killed between commits resumes the last commit.\nThe configurable health interval reduces CPU/GPU synchronization; a failure\nrolls back every parameter, moment, RNG, clock, and diagnostic row to that commit.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import asdict, replace\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport time\n\nimport numpy as np\nimport torch\n\ntry:\n    from .core import Config, Model, LogAdam, acquisition_rates, gradient_step, rates_at, export_dataset\n    from .runner import _validate, audit_native, clean, evaluate, fingerprint, snapshot, write_csv\n    from .asymptotics import evaluate_asymptotics\nexcept ImportError:\n    from core import Config, Model, LogAdam, acquisition_rates, gradient_step, rates_at, export_dataset\n    from runner import _validate, audit_native, clean, evaluate, fingerprint, snapshot, write_csv\n    from asymptotics import evaluate_asymptotics\n\n\nSCHEMA_VERSION = 1\nTRAINING_SOURCES = ("core.py", "runner.py", "a100_runner.py", "asymptotics.py")\nRESUMABLE_STATUSES = {"running", "finite_budget_complete", "time_budget_stop", "paused", "interrupted"}\n\n\ndef _sync(device):\n    if str(device).startswith("cuda"):\n        torch.cuda.synchronize(device)\n\n\ndef _atomic_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    with temporary.open("w") as handle:\n        json.dump(clean(value), handle, indent=2, allow_nan=False)\n        handle.write("\\n")\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, path)\n\n\ndef _atomic_torch_save(path, payload):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    with temporary.open("wb") as handle:\n        torch.save(payload, handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, path)\n\n\ndef _atomic_csv(path, rows):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    write_csv(temporary, rows)\n    os.replace(temporary, path)\n\n\ndef _source_hashes():\n    root = Path(__file__).resolve().parent\n    return {name: hashlib.sha256((root / name).read_bytes()).hexdigest() for name in TRAINING_SOURCES}\n\n\ndef _scientific_config(config):\n    return {key: value for key, value in asdict(config).items() if key not in ("steps", "max_seconds")}\n\n\ndef _cpu_tensors(values):\n    return [value.detach().cpu().clone() for value in values]\n\n\ndef _optimizer_state(optimizer):\n    if optimizer is None:\n        return None\n    return dict(t=optimizer.t, b1=optimizer.b1, b2=optimizer.b2,\n                ms=_cpu_tensors(optimizer.ms), ml=_cpu_tensors(optimizer.ml), vl=_cpu_tensors(optimizer.vl))\n\n\ndef _restore(payload, model, optimizer, rng):\n    with torch.no_grad():\n        for parameter, saved in zip(model.params(), payload["model"]):\n            parameter.copy_(saved.to(device=parameter.device, dtype=parameter.dtype))\n    state = payload["optimizer"]\n    if optimizer is not None:\n        if state is None or state["b1"] != optimizer.b1 or state["b2"] != optimizer.b2:\n            raise ValueError("Checkpoint optimizer does not match the configured Adam")\n        optimizer.t = state["t"]\n        for name in ("ms", "ml", "vl"):\n            setattr(optimizer, name, [saved.to(device=parameter.device, dtype=parameter.dtype).clone()\n                                     for saved, parameter in zip(state[name], model.params())])\n    elif state is not None:\n        raise ValueError("SGD checkpoint unexpectedly contains Adam moments")\n    rng.bit_generator.state = payload["rng_state"]\n\n\ndef _check_health(model, gradients, info, frozen_initial):\n    if not all(bool(torch.isfinite(parameter).all()) for parameter in model.params()):\n        return "nonfinite parameter"\n    if not math.isfinite(float(info["logloss"])):\n        return "nonfinite signed-log objective"\n    if not all(bool(torch.isfinite(sign).all()) and\n               bool(torch.where(sign != 0, torch.isfinite(logabs), torch.ones_like(sign, dtype=torch.bool)).all())\n               for sign, logabs in gradients):\n        return "nonfinite active signed-log gradient"\n    if float(info["table_log_span"]) > 650:\n        return "raw-table gradient aggregation exceeds 650 log units"\n    frozen = list(model.frozen_scalar_indices)\n    if frozen and not torch.equal(model.theta[frozen], frozen_initial[frozen]):\n        return "frozen gate coordinate changed"\n    return None\n\n\ndef _gradient_equivalence(eager, compiled, tolerance=1e-7):\n    """Compare signs and logarithmic magnitudes without exponent underflow."""\n    diagnostics = []\n    for (es, el), (cs, cl) in zip(eager, compiled):\n        active = es != 0\n        signs_equal = bool(torch.equal(es, cs))\n        discrepancy = torch.where(active, torch.abs(el - cl), torch.zeros_like(el))\n        max_error = float(discrepancy.max())\n        diagnostics.append(dict(signs_equal=signs_equal, max_absolute_log_gradient_error=max_error))\n    if not all(row["signs_equal"] and math.isfinite(row["max_absolute_log_gradient_error"])\n               and row["max_absolute_log_gradient_error"] <= tolerance for row in diagnostics):\n        raise RuntimeError(f"Compiled gradient equivalence audit failed: {diagnostics}")\n    return diagnostics\n\n\nclass _GradientBackend:\n    def __init__(self, model, compiled=False):\n        self.model = model\n        self.compiled = compiled\n        self.function = None\n        self.audits = []\n        self.compile_seconds = 0.\n        self.shapes = set()\n\n    def __call__(self, counts=None):\n        if not self.compiled:\n            return self.model.gradients(counts, validate_counts=False)\n        if self.function is None:\n            if not hasattr(torch, "compile"):\n                raise RuntimeError("compile_gradients=True requires torch.compile; use False explicitly for eager execution")\n            self.function = torch.compile(lambda counts: self.model.gradients(counts, validate_counts=False),\n                                          fullgraph=True, mode="default")\n        key = "full_batch" if counts is None else (tuple(counts.shape), str(counts.dtype))\n        if key in self.shapes:\n            return self.function(counts)\n        # Compilation is deliberately opt-in, and must agree at the actual state.\n        expected, expected_info = self.model.gradients(counts, validate_counts=False)\n        _sync(self.model.cfg.device)\n        start = time.monotonic()\n        try:\n            actual, actual_info = self.function(counts)\n            _sync(self.model.cfg.device)\n        except Exception as exc:\n            raise RuntimeError("Compiled gradients failed. Eager fallback is not automatic; restart with compile_gradients=False in a new run directory.") from exc\n        self.compile_seconds += time.monotonic() - start\n        audit = _gradient_equivalence(expected, actual)\n        loss_error = abs(float(expected_info["logloss"] - actual_info["logloss"]))\n        if not loss_error <= 1e-9:\n            raise RuntimeError(f"Compiled objective differs from eager objective: log-loss discrepancy {loss_error}")\n        self.audits.append(dict(input_kind=str(key), blocks=audit, absolute_log_loss_error=loss_error))\n        self.shapes.add(key)\n        return actual, actual_info\n\n\ndef _step_inputs(config, kind, step, acquisition, rng):\n    if step <= 3:\n        return 0., acquisition[step - 1], None, config.n * (config.n - 1)\n    tail = step - 4\n    base, rates = rates_at(config, kind, tail)\n    counts, batch = None, config.n * (config.n - 1)\n    if kind == "sgd":\n        batch = math.ceil(config.pair_batch * (1 + tail / config.offset) ** config.batch_growth)\n        # Aggregated iid sampling is exact, and the RNG state is checkpointed.\n        probs = np.full(config.n * (config.n - 1), 1 / (config.n * (config.n - 1)))\n        counts = torch.as_tensor(rng.multinomial(batch, probs), device=config.device, dtype=torch.float64)\n    return base, rates, counts, batch\n\n\ndef _optimizer_step(model, optimizer, gradients, rates, step):\n    if optimizer is None:\n        gradient_step(model, gradients, rates)\n    else:\n        c = model.cfg\n        optimizer.step(model.params(), gradients, rates, 3 * math.log(c.sigma) - c.eps_decay * (step - 1))\n\n\ndef _moment_diagnostics(model, optimizer, step, S):\n    q, p, u, v, x, w, m, g, h, rho, lrho = [float(value) for value in model.quantities()]\n    row = {name: value / S if S > 0 else None\n           for name, value in (("q_over_S", q), ("p_over_S", p), ("u_over_S", u),\n                               ("v_over_S", v), ("x_over_S", x), ("h_over_S", h), ("w_over_S", w))}\n    row.update(m_over_S_squared=m / S ** 2 if S > 0 else None,\n               g_over_S_squared=g / S ** 2 if S > 0 else None)\n    if optimizer is None or optimizer.t == 0:\n        return row\n    logeps = 3 * math.log(model.cfg.sigma) - model.cfg.eps_decay * (step - 1)\n    corrm = math.log1p(-optimizer.b1 ** optimizer.t)\n    corrv = math.log1p(-optimizer.b2 ** optimizer.t)\n    log_rms = .5 * (optimizer.vl[-1] - corrv)\n    denom = torch.logaddexp(log_rms, torch.full_like(log_rms, logeps))\n    direction = torch.where(optimizer.ms[-1] == 0, 0.,\n                            optimizer.ms[-1] * torch.exp(optimizer.ml[-1] - corrm - denom))\n    row.update(adam_t=optimizer.t, adam_log_epsilon=logeps)\n    gradient_signs, gradient_logs = model.gradients(validate_counts=False)[0][-1]\n    active = [i for i in range(6) if i not in model.frozen_scalar_indices]\n    factors = [i for i in range(4) if i not in model.frozen_scalar_indices]\n    row["adam_scalar_epsilon_dominated_fraction"] = float((logeps > log_rms[active]).double().mean())\n    row["adam_factor_epsilon_dominated_fraction"] = float((logeps > log_rms[factors]).double().mean())\n    for i, name in enumerate(("q", "p", "u", "v", "x", "w")):\n        row[f"adam_log_rms_{name}"] = float(log_rms[i])\n        row[f"adam_log_epsilon_over_rms_{name}"] = logeps - float(log_rms[i])\n        row[f"adam_direction_{name}"] = float(direction[i])\n        row[f"adam_log_abs_mhat_over_sqrt_v_{name}"] = (\n            float(optimizer.ml[-1][i] - corrm - log_rms[i]) if float(optimizer.ms[-1][i]) != 0 else None)\n        row[f"adam_log_grad_over_sqrt_v_{name}"] = (\n            float(gradient_logs[i] - log_rms[i]) if float(gradient_signs[i]) != 0 else None)\n        row[f"adam_population_gradient_sign_{name}"] = float(gradient_signs[i])\n    return row\n\n\ndef _branch_summary(payload, requested_steps):\n    state = payload["state"]\n    return dict(**payload["tags"], completed_steps=state["step"], requested_steps=requested_steps,\n                status=state["status"], reason=state["reason"], initial_hash=payload["initial_hash"],\n                final_hash=fingerprint(payload["model"]), acquired_positive_gaps=state["acquired"],\n                train_R=payload["config"]["R"], S=state["S"],\n                elapsed_seconds=state["elapsed_seconds"], examples=state["examples"],\n                updates_per_second=state["step"] / state["elapsed_seconds"] if state["elapsed_seconds"] > 0 else None,\n                acquired_reference_gap=state["reference_gap"], resume_count=state["resume_count"],\n                failed_at_step=state.get("failed_at_step"), checkpoint_schema=SCHEMA_VERSION,\n                compile_seconds=state.get("compile_seconds", 0.))\n\n\ndef _branch_record(payload, requested_steps):\n    """Lightweight derived ledger; model/optimizer tensors stay in the .pt file."""\n    return dict(summary=_branch_summary(payload, requested_steps),\n                history=payload["history"], probabilities=payload["probabilities"],\n                errors=payload["errors"], radii=payload["radii"], audits=payload["audits"],\n                compile_audits=payload.get("compile_audits", []))\n\n\ndef _aggregate(out, requested_steps, records_cache):\n    names = {"history.csv": "history", "lag_probabilities.csv": "probabilities",\n             "asymptotic_errors.csv": "errors", "certified_radii.csv": "radii"}\n    collected = {name: [] for name in names}\n    runs, audits, compile_audits = [], [], []\n    for label in sorted(records_cache):\n        payload = records_cache[label]\n        runs.append({**payload["summary"], "requested_steps": requested_steps})\n        for filename, field in names.items():\n            collected[filename].extend(payload[field])\n        audits.extend(payload["audits"])\n        compile_audits.extend(payload.get("compile_audits", []))\n    for filename, rows in collected.items():\n        _atomic_csv(out / filename, rows)\n    _atomic_csv(out / "runs.csv", runs)\n    _atomic_json(out / "native_gradient_audits.json", audits)\n    _atomic_json(out / "compiled_gradient_audits.json", compile_audits)\n\n\ndef run_large_suite(cfg, out_dir, seeds=(0, 1, 2), gate_modes=("learned", "retrieval_frozen"),\n                    optimizers=("sgd", "adam"),\n                    eval_lags=(1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024),\n                    prefixes=(0, 64), log_every=1000, checkpoint_every=1000,\n                    resume=True, progress=True, fixed_lags=(1, 2, 3, 4, 5, 8, 16, 32, 64),\n                    theta_values=(.25, .5, .75), clock_coefficients=(.001, .003, .01),\n                    error_targets=(.1, .01, .001), check_every=100,\n                    compile_gradients=False, max_updates=None):\n    """Run or resume a complete matrix without resetting any training state.\n\n    Only ``cfg.steps`` and ``cfg.max_seconds`` may increase on resume. Reporting\n    source changes are allowed; core, training, and asymptotic-evaluator sources\n    must match their original hashes. ``max_updates`` is an optional operational\n    pause budget for this invocation, useful for managed sessions and QA.\n    """\n    _validate(cfg)\n    if any(type(value) is not int or value < 1 for value in (log_every, checkpoint_every, check_every)):\n        raise ValueError("Logging, checkpoint, and health intervals must be positive integers")\n    if max_updates is not None and (type(max_updates) is not int or max_updates < 1):\n        raise ValueError("max_updates must be a positive integer or None")\n    if not seeds or any(type(seed) is not int or seed < 0 for seed in seeds) or len(set(seeds)) != len(seeds):\n        raise ValueError("Use distinct nonnegative integer seeds")\n    if not gate_modes or not set(gate_modes) <= {"learned", "retrieval_frozen", "both_frozen"}:\n        raise ValueError("Unknown or empty gate modes")\n    if not optimizers or not set(optimizers) <= {"sgd", "adam"}:\n        raise ValueError("Use sgd and/or adam")\n    if len(set(gate_modes)) != len(gate_modes) or len(set(optimizers)) != len(optimizers):\n        raise ValueError("Do not repeat matrix branches")\n    if not eval_lags or any(type(lag) is not int or lag < 1 for lag in eval_lags):\n        raise ValueError("Use positive integer evaluation lags")\n    if not prefixes or any(type(prefix) is not int or prefix < 0 for prefix in prefixes):\n        raise ValueError("Use nonnegative integer finite prefixes")\n    for coefficients in (theta_values, clock_coefficients, error_targets):\n        if not coefficients or any(not math.isfinite(value) or value <= 0 for value in coefficients):\n            raise ValueError("Asymptotic coefficients and error targets must be positive and finite")\n    out = Path(out_dir).resolve()\n    hashes = _source_hashes()\n    options = clean(dict(seeds=seeds, gate_modes=gate_modes, optimizers=optimizers,\n                         eval_lags=eval_lags, prefixes=prefixes, log_every=log_every,\n                         checkpoint_every=checkpoint_every, check_every=check_every,\n                         fixed_lags=fixed_lags, theta_values=theta_values,\n                         clock_coefficients=clock_coefficients, error_targets=error_targets,\n                         compile_gradients=compile_gradients))\n    manifest_path = out / "config.json"\n    if manifest_path.exists():\n        if not resume:\n            raise FileExistsError("Existing run requires resume=True or a new output directory")\n        saved = json.loads(manifest_path.read_text())\n        if saved["schema_version"] != SCHEMA_VERSION:\n            raise ValueError("Unsupported checkpoint schema")\n        if saved["scientific_config"] != clean(_scientific_config(cfg)) or saved["options"] != options:\n            raise ValueError("Scientific configuration or diagnostic protocol changed; start a new run")\n        if saved["source_sha256"] != hashes:\n            changed = [name for name, digest in hashes.items() if saved["source_sha256"].get(name) != digest]\n            raise ValueError(f"Training source changed since checkpoint: {changed}; use the saved source snapshot or start a new run")\n        if cfg.steps < saved["config"]["steps"] or cfg.max_seconds < saved["config"]["max_seconds"]:\n            raise ValueError("Resume permits extending steps/max_seconds, not shrinking their declared budgets")\n        saved["config"] = asdict(cfg)\n        saved["budget_history"].append(dict(steps=cfg.steps, max_seconds=cfg.max_seconds))\n    else:\n        if out.exists() and any(out.iterdir()):\n            raise FileExistsError("Nonempty output directory has no compatible long-run manifest")\n        out.mkdir(parents=True, exist_ok=True)\n        (out / "checkpoints").mkdir()\n        (out / "branch_records").mkdir()\n        (out / "source").mkdir()\n        for name in TRAINING_SOURCES:\n            (out / "source" / name).write_bytes((Path(__file__).resolve().parent / name).read_bytes())\n        export_dataset(cfg, out / "dataset")\n        saved = dict(schema_version=SCHEMA_VERSION, config=asdict(cfg), options=options,\n                     scientific_config=_scientific_config(cfg), source_sha256=hashes,\n                     budget_history=[dict(steps=cfg.steps, max_seconds=cfg.max_seconds)],\n                     torch_version=torch.__version__, numpy_version=np.__version__,\n                     python_version=platform.python_version(), cuda_available=torch.cuda.is_available(),\n                     device_name=torch.cuda.get_device_name(cfg.device) if str(cfg.device).startswith("cuda") else "CPU",\n                     numerical_backend="float64 signed-log exact gradients and Adam moments; fullgraph compilation opt-in",\n                     protocol="three ordinary-answer acquisition steps, practical polynomial continuation, no moment reset",\n                     resume_policy="All scientific settings and training source hashes fixed; only steps/max_seconds may increase",\n                     recovery_policy="A numerical failure or interrupted update rolls back to the last atomic validated checkpoint",\n                     source_policy="core.py, runner.py, a100_runner.py, asymptotics.py pinned; reporting/notebook source excluded",\n                     timing_policy="Cumulative branch compute/evaluation time across resumes; checkpoint I/O separately reported",\n                     limits=["Finite runs cannot prove convergence as t tends to infinity.",\n                             "Polynomial SGD continuation is practical, not the proof\'s existential envelope schedule.",\n                             "Frozen answer theorem covers R=2; larger R is an empirical extension.",\n                             "Finite-prefix probes do not exhaust all streams; certified bounds are separate."])\n    _atomic_json(manifest_path, saved)\n    (out / "branch_records").mkdir(exist_ok=True)\n    invocation_updates = 0\n    initial_hash_by_seed = {}\n    records_cache = {}\n    for path in (out / "checkpoints").glob("*.pt"):\n        existing = torch.load(path, map_location="cpu", weights_only=False)\n        seed = existing["tags"]["seed"]\n        if seed in initial_hash_by_seed and initial_hash_by_seed[seed] != existing["initial_hash"]:\n            raise ValueError("Saved paired initializations disagree")\n        initial_hash_by_seed[seed] = existing["initial_hash"]\n        # Once per invocation, repair derived records from authoritative state.\n        records_cache[path.stem] = _branch_record(existing, cfg.steps)\n        _atomic_json(out / "branch_records" / f"{path.stem}.json", records_cache[path.stem])\n\n    for seed in seeds:\n        for gate_mode in gate_modes:\n            for kind in optimizers:\n                c = replace(cfg, seed=seed, gate_mode=gate_mode)\n                tags = dict(seed=seed, gate_mode=gate_mode, optimizer=kind)\n                label = f"seed{seed}_{gate_mode}_{kind}"\n                checkpoint_path = out / "checkpoints" / f"{label}.pt"\n                model = Model(c)\n                initial_hash = fingerprint(model.params())\n                initial_hash_by_seed.setdefault(seed, initial_hash)\n                if initial_hash_by_seed[seed] != initial_hash:\n                    raise AssertionError("Initial tensors differ between matched branches")\n                optimizer = LogAdam(model.params(), c.beta1, c.beta2) if kind == "adam" else None\n                rng = np.random.default_rng(seed + 104729)\n                acquisition = acquisition_rates(c, kind)\n                if checkpoint_path.exists():\n                    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n                    if payload["initial_hash"] != initial_hash or payload["source_sha256"] != hashes:\n                        raise ValueError("Checkpoint provenance does not match this branch")\n                    if payload["state"]["status"] not in RESUMABLE_STATUSES:\n                        if progress:\n                            print(f"{label}: retained {payload[\'state\'][\'status\']} (not resumed)", flush=True)\n                        continue\n                    if payload["state"]["step"] >= c.steps:\n                        if progress:\n                            print(f"{label}: already completed {payload[\'state\'][\'step\']} steps", flush=True)\n                        continue\n                    if payload["state"]["elapsed_seconds"] >= c.max_seconds:\n                        if progress:\n                            print(f"{label}: cumulative time budget exhausted; increase max_seconds to continue", flush=True)\n                        continue\n                    _restore(payload, model, optimizer, rng)\n                    payload["state"]["resume_count"] += 1\n                    payload["state"]["status"], payload["state"]["reason"] = "running", ""\n                else:\n                    payload = dict(schema_version=SCHEMA_VERSION, config=asdict(c), tags=tags,\n                                   source_sha256=hashes, initial_hash=initial_hash,\n                                   initial_tables=_cpu_tensors(model.params()[:2]),\n                                   frozen_initial=model.theta.detach().cpu().clone(),\n                                   history=[], probabilities=[], errors=[], radii=[], audits=[], compile_audits=[],\n                                   state=dict(step=0, S=0., examples=0, elapsed_seconds=0., status="running", reason="",\n                                              acquired=None, reference_gap=None, resume_count=0, compile_seconds=0.,\n                                              checkpoint_write_seconds=0.))\n                    audit = audit_native(model)\n                    payload["audits"].append(dict(**tags, step=0, **audit))\n                    if not audit["passed"]:\n                        raise AssertionError(f"Initial ordinary-loss audit failed: {label}: {audit}")\n                state = payload["state"]\n                initial_tables = [value.to(c.device) for value in payload["initial_tables"]]\n                frozen_initial = payload["frozen_initial"].to(c.device)\n                elapsed_before = state["elapsed_seconds"]\n                session_start = time.monotonic()\n                backend = _GradientBackend(model, compiled=compile_gradients)\n                paused = False\n\n                def elapsed():\n                    return elapsed_before + time.monotonic() - session_start\n\n                def record():\n                    # Replace rows for this exact step if a budget boundary is revisited.\n                    for name in ("history", "probabilities", "errors", "radii"):\n                        payload[name] = [row for row in payload[name] if row["step"] != state["step"]]\n                    row = snapshot(model, tags, state["step"], state["S"], initial_tables, state["examples"], elapsed())\n                    row.update(_moment_diagnostics(model, optimizer, state["step"], state["S"]))\n                    row.update(acquired_reference_gap=state["reference_gap"],\n                               updates_per_second=state["step"] / elapsed() if elapsed() > 0 else None,\n                               compile_gradients=compile_gradients)\n                    payload["history"].append(row)\n                    payload["probabilities"].extend(evaluate(model, tags, state["step"], state["S"], eval_lags, prefixes))\n                    errors, radii = evaluate_asymptotics(model, tags, state["step"], state["S"],\n                        reference_gap=state["reference_gap"], fixed_lags=fixed_lags, theta_values=theta_values,\n                        clock_coefficients=clock_coefficients, error_targets=error_targets)\n                    payload["errors"].extend(errors)\n                    payload["radii"].extend(radii)\n                    if progress:\n                        print(f"{label}: {state[\'step\']}/{c.steps}, loss={row[\'loss\']:.4g}, "\n                              f"S={state[\'S\']:.4g}, content horizon={row[\'content_horizon\']:.3g}", flush=True)\n\n                def commit():\n                    state["elapsed_seconds"] = elapsed()\n                    state["compile_seconds"] += backend.compile_seconds\n                    backend.compile_seconds = 0.\n                    payload["compile_audits"].extend(dict(**tags, step=state["step"], **row) for row in backend.audits)\n                    backend.audits.clear()\n                    payload.update(model=_cpu_tensors(model.params()), optimizer=_optimizer_state(optimizer),\n                                   rng_state=rng.bit_generator.state, config=asdict(c))\n                    start_write = time.monotonic()\n                    _atomic_torch_save(checkpoint_path, payload)\n                    state["checkpoint_write_seconds"] += time.monotonic() - start_write\n                    records_cache[label] = _branch_record(payload, c.steps)\n                    _atomic_json(out / "branch_records" / f"{label}.json", records_cache[label])\n                    _aggregate(out, c.steps, records_cache)\n\n                def rollback(reason, failed_step, status="numerical_stop"):\n                    nonlocal payload, state\n                    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n                    _restore(payload, model, optimizer, rng)\n                    state = payload["state"]\n                    state.update(status=status, reason=reason, failed_at_step=failed_step)\n                    # The recovered model is the last committed validated state.\n                    payload["config"] = asdict(c)\n                    _atomic_torch_save(checkpoint_path, payload)\n                    records_cache[label] = _branch_record(payload, c.steps)\n                    _atomic_json(out / "branch_records" / f"{label}.json", records_cache[label])\n                    _aggregate(out, c.steps, records_cache)\n\n                if state["step"] == 0:\n                    record()\n                    commit()\n                try:\n                    for step in range(state["step"] + 1, c.steps + 1):\n                        if elapsed() >= c.max_seconds:\n                            gradients, info = model.gradients(validate_counts=False)\n                            reason = _check_health(model, gradients, info, frozen_initial)\n                            if reason:\n                                rollback(reason, step - 1)\n                            else:\n                                state.update(status="time_budget_stop", reason="cumulative per-branch time budget")\n                                record()\n                                commit()\n                            break\n                        base, rates, counts, batch = _step_inputs(c, kind, step, acquisition, rng)\n                        # Keep acquisition native; compile at the acquired state.\n                        gradients, info = (model.gradients(counts, validate_counts=False) if step <= 3 else backend(counts))\n                        _optimizer_step(model, optimizer, gradients, rates, step)\n                        state["step"], state["S"] = step, state["S"] + base\n                        state["examples"] += 4 * batch + 2 * c.n\n                        invocation_updates += 1\n                        should_pause = max_updates is not None and invocation_updates >= max_updates\n                        is_checkpoint = step <= 3 or step % checkpoint_every == 0 or step == c.steps or should_pause\n                        is_log = step == 3 or step % log_every == 0 or step == c.steps or should_pause\n                        if step % check_every == 0 or is_checkpoint or is_log:\n                            health_gradients, health_info = model.gradients(validate_counts=False)\n                            reason = _check_health(model, health_gradients, health_info, frozen_initial)\n                            if reason:\n                                rollback(reason, step)\n                                break\n                        if step == 3:\n                            state["reference_gap"] = float(model.gaps()[model.mask].min())\n                            state["acquired"] = state["reference_gap"] > 0 and float(model.theta[-1]) > 0\n                            audit = audit_native(model)\n                            payload["audits"].append(dict(**tags, step=3, **audit))\n                            if not audit["passed"]:\n                                rollback(f"Acquisition native audit failed: {audit}", step)\n                                break\n                            if not state["acquired"]:\n                                state.update(status="acquisition_failed", reason="three ordinary updates did not acquire every positive gap and decoder")\n                                record()\n                                commit()\n                                break\n                        if step == c.steps:\n                            state.update(status="finite_budget_complete", reason="")\n                        elif should_pause:\n                            state.update(status="paused", reason="max_updates operational pause")\n                            paused = True\n                        if is_log:\n                            record()\n                        if is_checkpoint:\n                            commit()\n                        if should_pause:\n                            break\n                except KeyboardInterrupt:\n                    rollback("Interrupted between atomic checkpoints; restored committed state", state["step"], "interrupted")\n                    raise\n                except Exception:\n                    # Do not commit a possibly half-updated state after an exception.\n                    rollback("Exception between atomic checkpoints; restored committed state", state["step"], "interrupted")\n                    raise\n                if progress:\n                    print(f"{label}: {state[\'status\']} at committed step {state[\'step\']}", flush=True)\n                if paused or (max_updates is not None and invocation_updates >= max_updates):\n                    return str(out)\n    _aggregate(out, cfg.steps, records_cache)\n    return str(out)\n\n\ndef benchmark_config(cfg, steps=30, warmup=5, num_seeds=3, compile_gradients=False,\n                     gate_modes=("learned", "retrieval_frozen"), optimizers=("sgd", "adam")):\n    """Measure local hardware throughput using fresh models, never run state.\n\n    Excludes compilation, warmup, evaluation, and checkpoint I/O from steady\n    training timing; reports compilation separately. Matrix estimates are\n    estimates, not promises, and exclude diagnostic and storage overhead.\n    """\n    _validate(cfg)\n    if type(steps) is not int or steps < 1 or type(warmup) is not int or warmup < 0 or num_seeds < 1:\n        raise ValueError("Use positive benchmark steps/seed count and nonnegative warmup")\n    rows = []\n    for gate in gate_modes:\n        for kind in optimizers:\n            c = replace(cfg, seed=0, gate_mode=gate)\n            model = Model(c)\n            optimizer = LogAdam(model.params(), c.beta1, c.beta2) if kind == "adam" else None\n            rng = np.random.default_rng(104729)\n            acquisition = acquisition_rates(c, kind)\n            start_acquisition = time.monotonic()\n            for step in range(1, 4):\n                _, rates, counts, _ = _step_inputs(c, kind, step, acquisition, rng)\n                _optimizer_step(model, optimizer, model.gradients(counts, validate_counts=False)[0], rates, step)\n            _sync(c.device)\n            acquisition_seconds = time.monotonic() - start_acquisition\n            native = audit_native(model)\n            if not native["passed"]:\n                raise AssertionError(f"Benchmark acquired native audit failed: {native}")\n            acquired_gap = float(model.gaps()[model.mask].min())\n            backend = _GradientBackend(model, compiled=compile_gradients)\n            # At least one untimed call compiles the exact backend shape.\n            for step in range(4, 4 + max(1, warmup)):\n                _, rates, counts, _ = _step_inputs(c, kind, step, acquisition, rng)\n                _optimizer_step(model, optimizer, backend(counts)[0], rates, step)\n            _sync(c.device)\n            if str(c.device).startswith("cuda"):\n                torch.cuda.reset_peak_memory_stats(c.device)\n            start = time.monotonic()\n            for step in range(4 + max(1, warmup), 4 + max(1, warmup) + steps):\n                _, rates, counts, _ = _step_inputs(c, kind, step, acquisition, rng)\n                _optimizer_step(model, optimizer, backend(counts)[0], rates, step)\n            _sync(c.device)\n            seconds = time.monotonic() - start\n            gradients, info = model.gradients(validate_counts=False)\n            if not all(bool(torch.isfinite(parameter).all()) for parameter in model.params()):\n                raise FloatingPointError("Benchmark produced nonfinite parameters")\n            per_step = seconds / steps\n            rows.append(dict(gate_mode=gate, optimizer=kind, measured_steps=steps,\n                             measured_training_seconds=seconds, seconds_per_step=per_step,\n                             updates_per_second=1 / per_step, acquisition_seconds=acquisition_seconds,\n                             compile_seconds=backend.compile_seconds, compiled_audits=backend.audits,\n                             acquired_minimum_gap=acquired_gap, acquired_positive_gaps=acquired_gap > 0,\n                             estimated_branch_seconds=acquisition_seconds + max(0, cfg.steps - 3) * per_step,\n                             cuda_peak_allocated_bytes=torch.cuda.max_memory_allocated(c.device) if str(c.device).startswith("cuda") else None))\n    estimated = num_seeds * sum(row["estimated_branch_seconds"] for row in rows)\n    return clean(dict(device=cfg.device,\n                      device_name=torch.cuda.get_device_name(cfg.device) if str(cfg.device).startswith("cuda") else "CPU",\n                      n=cfg.n, d=cfg.d, R=cfg.R, requested_steps=cfg.steps, num_seeds=num_seeds,\n                      compile_gradients=compile_gradients, rows=rows,\n                      estimated_matrix_seconds=estimated, estimated_matrix_hours=estimated / 3600,\n                      caveat="Measured on this runtime; estimates exclude evaluations/checkpoints and long-run changes in sampling batch cost."))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out", required=True)\n    parser.add_argument("--n", type=int, default=128)\n    parser.add_argument("--d", type=int, default=4096)\n    parser.add_argument("--R", type=int, default=2)\n    parser.add_argument("--steps", type=int, default=200000)\n    parser.add_argument("--device", default="cuda")\n    parser.add_argument("--max-seconds", type=float, default=86400.)\n    parser.add_argument("--seeds", nargs="+", type=int, default=[0, 1, 2])\n    parser.add_argument("--compile-gradients", action="store_true")\n    parser.add_argument("--moment-preset", choices=("paper_short_memory", "standard_memory"), default="paper_short_memory")\n    parser.add_argument("--benchmark-only", action="store_true")\n    args = parser.parse_args()\n    if args.device == "cpu":\n        torch.set_num_threads(1)\n    beta1, beta2 = (.1, .1) if args.moment_preset == "paper_short_memory" else (.9, .999)\n    cfg = Config(n=args.n, d=args.d, R=args.R, steps=args.steps, device=args.device,\n                 max_seconds=args.max_seconds, beta1=beta1, beta2=beta2)\n    if args.benchmark_only:\n        print(json.dumps(benchmark_config(cfg, num_seeds=len(args.seeds), compile_gradients=args.compile_gradients), indent=2))\n    else:\n        run_large_suite(cfg, args.out, seeds=tuple(args.seeds), compile_gradients=args.compile_gradients)\n\n\nif __name__ == "__main__":\n    main()\n'


In [ ]:
SOURCES['asymptotic_report'] = '"""Plot certified brackets for worst-case answer error, using stored log errors.\n\nThe unknown supremum E_t(r) is bracketed by an analytic witness lower bound\nand a uniform upper certificate. Neither endpoint is an empirical estimate of\nthat supremum. Invalid certificates are displayed with the trivial upper 1.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom matplotlib.lines import Line2D\nfrom matplotlib.ticker import MaxNLocator\nimport numpy as np\n\n\nARMS = (("learned", "sgd"), ("learned", "adam"),\n        ("retrieval_frozen", "sgd"), ("retrieval_frozen", "adam"))\nGATES = {"learned": "Learned retrieval", "retrieval_frozen": "Frozen retrieval"}\nPALETTE = tuple(plt.get_cmap("tab10").colors)\n\n\ndef _float(value, default=math.nan):\n    try:\n        return float(value)\n    except (ValueError, TypeError):\n        return default\n\n\ndef _read(path):\n    if not path.exists() or not path.stat().st_size:\n        return []\n    with path.open(newline="", encoding="utf-8") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef _valid(row):\n    return str(row.get("bound_valid", "")).lower() in {"true", "1", "1.0", "yes"}\n\n\ndef _fixed(row):\n    return str(row.get("probe", "")).lower().startswith("fixed")\n\n\ndef _probe(row):\n    name = str(row.get("probe", "unknown"))\n    if _fixed(row):\n        return f"fixed r={_float(row.get(\'lag\')):g}"\n    parameter = row.get("theta")\n    if parameter not in (None, ""):\n        return f"{name}, θ={_float(parameter):g}"\n    parameter = row.get("coefficient")\n    if parameter not in (None, ""):\n        return f"{name}, c={_float(parameter):g}"\n    return name\n\n\ndef _curves(rows):\n    grouped = defaultdict(list)\n    for row in rows:\n        grouped[(str(row.get("gate_mode")), str(row.get("optimizer")),\n                 str(row.get("seed")), _probe(row))].append(row)\n    return {key: sorted(values, key=lambda row: _float(row.get("step"), -1))\n            for key, values in grouped.items()}\n\n\ndef _log_error(row, side):\n    if side == "upper" and not _valid(row):\n        return 0.0\n    value = _float(row.get(f"log_{side}_error"))\n    if math.isnan(value):\n        raw = _float(row.get(f"{side}_error"))\n        if raw > 0:\n            value = math.log(raw)\n    return value / math.log(10)\n\n\ndef _save(fig, path):\n    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")\n    fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")\n    plt.close(fig)\n\n\ndef _axes_style(ax):\n    ax.grid(alpha=.18)\n    ax.spines[["top", "right"]].set_visible(False)\n\n\ndef _clock_axis(ax, xkey="S"):\n    threshold = 1 if xkey == "S" else 10\n    largest = ax.dataLim.xmax\n    if math.isfinite(largest) and largest > 2 * threshold:\n        ax.set_xscale("symlog", linthresh=threshold)\n    ax.set_xlim(left=0)\n\n\ndef _error_plot(rows, fixed, xkey, path):\n    rows = [row for row in rows if _fixed(row) == fixed]\n    grouped = _curves(rows)\n    probes = sorted({_probe(row) for row in rows})\n    colors = {probe: PALETTE[i % len(PALETTE)] for i, probe in enumerate(probes)}\n    fig, axes = plt.subplots(2, 2, figsize=(13.4, 8.5), squeeze=False)\n    for ax, arm in zip(axes.ravel(), ARMS):\n        seen = set()\n        has_invalid = False\n        for (gate, opt, seed, probe), values in grouped.items():\n            if (gate, opt) != arm:\n                continue\n            points = [(row, _float(row.get(xkey)), _log_error(row, "lower"), _log_error(row, "upper"))\n                      for row in values]\n            points = [p for p in points if math.isfinite(p[1])]\n            if not points:\n                continue\n            x = np.array([p[1] for p in points])\n            lower = np.array([p[2] for p in points])\n            upper = np.array([p[3] for p in points])\n            color = colors[probe]\n            ax.plot(x, upper, color=color, lw=1.5, alpha=.78,\n                    label=probe if probe not in seen else "_nolegend_")\n            ax.plot(x, lower, color=color, lw=1, ls=":", alpha=.75)\n            ax.fill_between(x, lower, upper, color=color, alpha=.055,\n                            where=np.isfinite(lower) & np.isfinite(upper))\n            invalid = [p[1] for p in points if not _valid(p[0])]\n            if invalid:\n                ax.scatter(invalid, np.zeros(len(invalid)), marker="x", color=color, s=16, zorder=5)\n                has_invalid = True\n            seen.add(probe)\n        if not seen:\n            ax.text(.5, .5, "No saved bracket measurements", transform=ax.transAxes, ha="center", color=".4")\n        else:\n            handles, labels = ax.get_legend_handles_labels()\n            handles += [Line2D([], [], color=".3", lw=1.6), Line2D([], [], color=".3", ls=":")]\n            labels += ["Upper endpoint", "Witness lower endpoint"]\n            if has_invalid:\n                handles.append(Line2D([], [], color=".3", marker="x", ls=""))\n                labels.append("Uncertified: trivial upper = 1")\n            ax.legend(handles, labels, fontsize=7, ncol=2, frameon=False, loc="best")\n        _clock_axis(ax, xkey)\n        ax.set(title=f"{GATES[arm[0]]} · {arm[1].upper()}",\n               xlabel="Continuation learning-rate sum S" if xkey == "S" else "Optimizer update t",\n               ylabel="log₁₀(error bound)")\n        ax.set_ylim(top=.15)\n        ax.axhline(-2, color=".6", ls="--", lw=.6)\n        _axes_style(ax)\n    subject = "Fixed radii: Eₜ(r)" if fixed else "Moving radii: Eₜ(rₜ)"\n    fig.suptitle(subject + " bracketed by a witness and a uniform certificate", fontsize=14)\n    fig.text(.5, .016, "Each seed is a separate envelope, not a seed confidence interval. "\n             "E is unknown inside the bracket; log errors are used directly without subtracting a probability from 1.",\n             ha="center", fontsize=8)\n    fig.tight_layout(rect=(0, .045, 1, .95))\n    _save(fig, path)\n\n\ndef _radius_plot(errors, radii, path):\n    fig, axes = plt.subplots(4, 2, figsize=(13.4, 13.5), squeeze=False)\n    moving = [row for row in errors if not _fixed(row)]\n    probes = sorted({_probe(row) for row in moving})\n    colors = {probe: PALETTE[i % len(PALETTE)] for i, probe in enumerate(probes)}\n    for row_index, arm in enumerate(ARMS):\n        ax = axes[row_index, 0]\n        seen = set()\n        for (gate, opt, seed, probe), values in _curves(moving).items():\n            if (gate, opt) != arm:\n                continue\n            points = [(_float(row.get("S")), _float(row.get("lag"))) for row in values]\n            points = [p for p in points if all(math.isfinite(v) for v in p)]\n            if points:\n                ax.plot(*zip(*points), color=colors[probe], alpha=.7, lw=1.3,\n                        label=probe if probe not in seen else "_nolegend_")\n                seen.add(probe)\n        ax.set(title=f"{GATES[arm[0]]} · {arm[1].upper()} — actual probe radius",\n               xlabel="Continuation sum S", ylabel="Actual integer radius rₜ")\n        if seen:\n            ax.legend(fontsize=7, ncol=2, frameon=False)\n        else:\n            ax.text(.5, .5, "No moving-radius observations", transform=ax.transAxes, ha="center", color=".4")\n        ax = axes[row_index, 1]\n        threshold_groups = defaultdict(list)\n        for row in radii:\n            if (row.get("gate_mode"), row.get("optimizer")) == arm:\n                threshold_groups[(str(row.get("seed")), str(row.get("error_target", row.get("threshold"))))].append(row)\n        seen = set()\n        has_uncertified = False\n        for (seed, threshold), values in sorted(threshold_groups.items()):\n            values.sort(key=lambda row: _float(row.get("step"), -1))\n            xs = [_float(row.get("S")) for row in values]\n            ys = [_float(row.get("certified_radius"), 0) for row in values]\n            threshold_index = sorted({key[1] for key in threshold_groups}).index(threshold)\n            color = PALETTE[threshold_index % len(PALETTE)]\n            ax.plot(xs, ys, color=color, alpha=.75, lw=1.4,\n                    label=f"E ≤ {_float(threshold):g}" if threshold not in seen else "_nolegend_")\n            invalid = [(x, y) for x, y, row in zip(xs, ys, values) if not _valid(row)]\n            if invalid:\n                ax.scatter(*zip(*invalid), color=color, marker="x", s=15)\n                has_uncertified = True\n            seen.add(threshold)\n        ax.set(title="Certified radius from uniform bound; no evaluation-lag cap",\n               xlabel="Continuation sum S", ylabel="Largest certified integer radius")\n        if seen:\n            if has_uncertified:\n                ax.plot([], [], color=".3", marker="x", ls="", label="Unavailable certificate")\n            ax.legend(fontsize=8, frameon=False)\n        else:\n            ax.text(.5, .5, "No saved certified radii", transform=ax.transAxes, ha="center", color=".4")\n        for ax in axes[row_index]:\n            _clock_axis(ax)\n            largest = ax.dataLim.ymax\n            if math.isfinite(largest) and largest > 16:\n                ax.set_yscale("symlog", linthresh=4)\n                ax.set_ylim(bottom=0)\n            else:\n                ax.set_ylim(0, max(1.1, 1.08 * largest) if math.isfinite(largest) else 1.1)\n                ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))\n            _axes_style(ax)\n    fig.suptitle("Measured radius trajectories and certified generalization range", fontsize=14)\n    fig.text(.5, .007, "Clock-based radii are chosen probes; adaptive radii use measured gaps. "\n             "Observed growth over a finite window does not establish rₜ → ∞. Radius 0 certifies no positive radius.",\n             ha="center", fontsize=8)\n    fig.tight_layout(rect=(0, .03, 1, .975))\n    _save(fig, path)\n\n\ndef _fmt(value):\n    value = _float(value)\n    if value == -math.inf:\n        return "−∞"\n    return f"{value:.5g}" if math.isfinite(value) else "unavailable"\n\n\ndef _optimizer_plot(history, config, path):\n    fig, axes = plt.subplots(2, 3, figsize=(16, 8), squeeze=False)\n    grouped = defaultdict(list)\n    for row in history:\n        grouped[(row.get("gate_mode"), row.get("optimizer"), str(row.get("seed")))].append(row)\n    columns = (("q", "am", 1.0), ("u", "ag", 1.5), ("w", "aw", 0.1))\n    for gate_index, gate in enumerate(("learned", "retrieval_frozen")):\n        for column_index, (parameter, multiplier_name, default) in enumerate(columns):\n            ax = axes[gate_index, column_index]\n            direction_ax = ax.twinx()\n            seen = set()\n            direction_seen = False\n            nominal = _float(config.get(multiplier_name), default)\n            for (row_gate, optimizer, seed), values in sorted(grouped.items()):\n                if row_gate != gate:\n                    continue\n                points, directions = [], []\n                for row in sorted(values, key=lambda item: _float(item.get("step"), -1)):\n                    S = _float(row.get("S"))\n                    if not math.isfinite(S) or S <= 0:\n                        continue\n                    ratio = _float(row.get(parameter + "_over_S"))\n                    if not math.isfinite(ratio):\n                        ratio = _float(row.get(parameter)) / S\n                    if math.isfinite(ratio):\n                        points.append((S, ratio))\n                    direction = _float(row.get("adam_direction_" + parameter))\n                    if optimizer == "adam" and math.isfinite(direction):\n                        directions.append((S, -direction))\n                if points:\n                    ax.plot(*zip(*points), color={"sgd": "#2166ac", "adam": "#d95f02"}.get(optimizer, ".3"),\n                            lw=1.5, alpha=.75,\n                            label=f"{optimizer.upper()} {parameter}/S" if optimizer not in seen else "_nolegend_")\n                    seen.add(optimizer)\n                if directions:\n                    direction_ax.plot(*zip(*directions), color="#762a83", ls="--", lw=1.2, alpha=.6,\n                                      label="Adam normalized growth direction" if not direction_seen else "_nolegend_")\n                    direction_seen = True\n            ax.axhline(nominal, color=".3", ls=":", lw=1,\n                       label=f"Adam nominal {multiplier_name} = {nominal:g}")\n            direction_ax.axhline(1, color="#762a83", ls=":", lw=.6, alpha=.4)\n            ax.set(title=f"{GATES[gate]} · {parameter}", xlabel="Continuation learning-rate sum S",\n                   ylabel=f"{parameter}/S")\n            direction_ax.set_ylabel("−m̂ / (√v̂ + ε)", color="#762a83")\n            direction_ax.tick_params(axis="y", colors="#762a83", labelsize=8)\n            direction_ax.spines["top"].set_visible(False)\n            direction_low, direction_high = direction_ax.dataLim.ymin, direction_ax.dataLim.ymax\n            direction_ax.set_ylim(min(0, 1.05 * direction_low) if math.isfinite(direction_low) else 0,\n                                  max(1.1, 1.05 * direction_high) if math.isfinite(direction_high) else 1.1)\n            direction_ax.ticklabel_format(axis="y", style="plain", useOffset=False)\n            if not direction_seen:\n                direction_ax.set_ylim(0, 1.1)\n                ax.text(.02, .03, "Adam directions unavailable in these logs", transform=ax.transAxes,\n                        fontsize=7, color=".4")\n            if not seen:\n                ax.text(.5, .5, "No saved positive-S ratio observations", transform=ax.transAxes,\n                        ha="center", fontsize=8, color=".4")\n            _clock_axis(ax)\n            _axes_style(ax)\n            handles, labels = ax.get_legend_handles_labels()\n            more_handles, more_labels = direction_ax.get_legend_handles_labels()\n            ax.legend(handles + more_handles, labels + more_labels, fontsize=7, frameon=False)\n    fig.suptitle("Optimizer tracking: cumulative parameter ratios and current Adam direction", fontsize=14)\n    fig.text(.5, .012, "Left axes: parameter/S, including the acquisition offset. Right axes: signed normalized growth direction. "\n             "Nominal references are conditional Adam asymptotic values, not finite-run guarantees.", ha="center", fontsize=8)\n    fig.tight_layout(rect=(0, .045, 1, .95))\n    _save(fig, path)\n\n\ndef _report(errors, radii, runs):\n    invalid_count = sum(not _valid(row) for row in errors)\n    lines = ["# Worst-case generalization error: finite-run bracket report", "",\n             "Define E_t(r) as the supremum of 1 − P_t(correct answer) over every finite record stream "\n             "whose latest assignment to the queried key is at lag at most r. This E is a model error, "\n             "not Adam\'s denominator epsilon.", "",\n             "The unknown E_t(r) is bounded below by an infinite-prefix witness supremum and above by "\n             "a conditional uniform certificate over arbitrary finite streams. The infinite-prefix witness "\n             "is a limit of finite streams and therefore supplies a valid lower bound on the supremum. "\n             "Finite-prefix sample minima and means are reported separately by the ordinary report.", "",\n             "Plots use the saved natural logarithms divided by log(10), so tiny errors remain visible "\n             "without evaluating 1 − a probability rounded to one. Solid and dotted curves are the upper "\n             "and lower endpoints for each seed. Their shaded interval brackets the true supremum; "\n             "it is not a confidence interval or a numerical estimate of E.", "",\n             f"Saved error rows: {len(errors)}. Rows without a valid uniform certificate: {invalid_count}. "\n             "Every invalid upper certificate is shown explicitly as the trivial upper bound E ≤ 1 "\n             "with a cross marker; no uncertified upper curve is silently omitted.", "",\n             "## What the trajectories test", "",\n             "- Fixed-radius probes examine error reduction at an unchanged lag threshold.",\n             "- Clock probes choose a radius proportional to S for learned retrieval and to S² for frozen "\n             "retrieval (with their saved coefficients and integer rounding). These prescribed clocks do "\n             "not assert that each optimizer supports that growth rate. In particular, frozen-SGD clock "\n             "probes are stress tests; its ordinary-answer theorem does not give a sharp time law.",\n             "- Adaptive probes use the measured content gap and forgetting scale. Their actual integer "\n             "radii are plotted. A decreasing upper error at a radius that stays bounded does not establish "\n             "generalization along a radius tending to infinity.",\n             "- Certified radii invert the uniform bound at each requested error threshold and are not "\n             "clipped to the largest finite evaluation lag. Radius zero means no positive radius is certified.",\n             "- R = 2 is the proved frozen-gate setting. The theoretical asymptotic conclusions require "\n             "their sufficient initialization, acquisition, continuation, and basin hypotheses. These "\n             "practical polynomial schedules are finite experiments, not literal implementations of the "\n             "existential conservative SGD schedule. Larger R is an extension experiment.",\n             "- The frozen intervention holds retrieval h fixed and continues training the binding gate. "\n             "Uniform frozen-gate success needs h₀ > log(2) and the certificate\'s other conditions.", "",\n             "`asymptotic_optimizer_diagnostics.png` compares q/S, u/S and w/S with their configured "\n             "nominal Adam multipliers, and separately shows the signed normalized Adam growth directions. "\n             "Ratios include finite acquisition offsets. Missing moment directions are labeled unavailable. "\n             "Changing a beta preset changes both acquisition and continuation dynamics, not only a later memory window.", "",\n             "## Latest saved brackets", "",\n             "Values are log10(error). A more negative upper endpoint is stronger evidence of a small "\n             "worst-case error at that saved radius; finite trajectories do not establish a limit.", "",\n             "| Seed | Gate | Optimizer | Probe | Step | S | Radius | log10 lower | log10 upper | Valid uniform bound |",\n             "| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |"]\n    for (gate, opt, seed, probe), values in sorted(_curves(errors).items()):\n        row = values[-1]\n        lines.append("| " + " | ".join([seed, gate, opt, probe, _fmt(row.get("step")), _fmt(row.get("S")),\n                                        _fmt(row.get("lag")), _fmt(_log_error(row, "lower")),\n                                        _fmt(_log_error(row, "upper")), str(_valid(row))]) + " |")\n    if not errors:\n        lines.append("| — | — | — | No saved data | — | — | — | unavailable | unavailable | False |")\n    lines.extend(["", "## Run status", "", "| Seed | Gate | Optimizer | Completed steps | Status | Reason |",\n                  "| --- | --- | --- | --- | --- | --- |"])\n    for row in runs:\n        lines.append("| " + " | ".join(str(row.get(key, "unavailable")).replace("|", "/").replace("\\n", " ")\n                                             for key in ("seed", "gate_mode", "optimizer", "completed_steps", "status", "reason")) + " |")\n    if not runs:\n        lines.append("| — | — | — | — | No saved status | — |")\n    lines.extend(["", "Each branch is plotted through its own last saved checkpoint. Stopped runs can have "\n                  "different budgets. Inspect `runs.csv`, `history.csv`, `asymptotic_errors.csv`, "\n                  "`certified_radii.csv`, and the checkpoint metadata before comparing arms.", ""])\n    return "\\n".join(lines)\n\n\ndef make_asymptotic_report(out_dir):\n    """Create standalone PNG/PDF figures and a Markdown interpretation guide."""\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    errors = _read(out_dir / "asymptotic_errors.csv")\n    radii = _read(out_dir / "certified_radii.csv")\n    runs = _read(out_dir / "runs.csv")\n    history = _read(out_dir / "history.csv")\n    config_path = out_dir / "config.json"\n    config_data = json.loads(config_path.read_text(encoding="utf-8")) if config_path.exists() else {}\n    config = config_data.get("config", config_data)\n    paths = {}\n    with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10}):\n        for fixed, label in ((True, "fixed"), (False, "moving")):\n            for xkey, xname in (("step", "step"), ("S", "S")):\n                name = f"asymptotic_{label}_error_by_{xname}"\n                path = out_dir / (name + ".png")\n                _error_plot(errors, fixed, xkey, path)\n                paths[name] = str(path)\n                paths[name + "_pdf"] = str(path.with_suffix(".pdf"))\n        path = out_dir / "asymptotic_radius_trajectories.png"\n        _radius_plot(errors, radii, path)\n        paths["radius_trajectories"] = str(path)\n        paths["radius_trajectories_pdf"] = str(path.with_suffix(".pdf"))\n        path = out_dir / "asymptotic_optimizer_diagnostics.png"\n        _optimizer_plot(history, config, path)\n        paths["optimizer_diagnostics"] = str(path)\n        paths["optimizer_diagnostics_pdf"] = str(path.with_suffix(".pdf"))\n    path = out_dir / "asymptotic_report.md"\n    path.write_text(_report(errors, radii, runs), encoding="utf-8")\n    paths["report"] = str(path)\n    return paths\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("out_dir", type=Path)\n    args = parser.parse_args()\n    print(json.dumps(make_asymptotic_report(args.out_dir), indent=2))\n'


In [ ]:
SOURCES['test_core'] = '"""Independent stream, derivative, optimizer, initialization, and dataset QA."""\nimport json\nimport math\nfrom pathlib import Path\nimport tempfile\nimport unittest\n\nimport torch\n\ntry:\n    from .core import (Config, Model, LogAdam, acquisition_rates, boundary_log_odds,\n                       dense_objective, dense_stream, export_dataset, gradient_step,\n                       native_gradients, native_loss, pair_cases, rates_at, slog)\nexcept ImportError:\n    from core import (Config, Model, LogAdam, acquisition_rates, boundary_log_odds,\n                      dense_objective, dense_stream, export_dataset, gradient_step,\n                      native_gradients, native_loss, pair_cases, rates_at, slog)\n\n\ndef moderate_model(R=2, gate_mode="learned", w=.8):\n    c = Config(n=3, d=5, R=R, gate_mode=gate_mode, D=.3, c0=1.2)\n    model = Model(c)\n    gen = torch.Generator().manual_seed(812)\n    model.Q = torch.randn(c.n, c.d, generator=gen, dtype=torch.float64) * .3\n    model.K = torch.randn(c.n, c.d, generator=gen, dtype=torch.float64) * .3\n    model.theta = torch.tensor([.8, .9, .65, .8, .4, w], dtype=torch.float64)\n    return model\n\n\nclass CoreTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(1)\n\n    def test_exact_stream_loss_and_gradients(self):\n        for R in (2, 3, 4, 7):\n            for gate_mode in ("learned", "retrieval_frozen", "both_frozen"):\n                for w in (-.6, 0., .8):\n                    for sampled in (False, True):\n                        with self.subTest(R=R, gate=gate_mode, w=w, sampled=sampled):\n                            model = moderate_model(R, gate_mode, w)\n                            counts = torch.tensor([0., 3., 1., 2., 0., 4.]) if sampled else None\n                            params = [p.clone().requires_grad_(True) for p in model.params()]\n                            loss_dense = dense_objective(model, counts, params)\n                            expected = torch.autograd.grad(loss_dense, params)\n                            native, info = native_gradients(model, counts)\n                            signed, slog_info = model.gradients(counts)\n                            torch.testing.assert_close(info["loss"], loss_dense, rtol=2e-12, atol=2e-14)\n                            torch.testing.assert_close(slog_info["logloss"].exp(), loss_dense, rtol=2e-12, atol=2e-14)\n                            for target, actual, (sign, logabs) in zip(expected, native, signed):\n                                self.assertFalse(bool(torch.isnan(logabs).any()))\n                                torch.testing.assert_close(actual, target, rtol=2e-10, atol=2e-13)\n                                torch.testing.assert_close(sign * logabs.exp(), target, rtol=2e-10, atol=2e-13)\n\n    def test_boundary_odds_match_independent_attention(self):\n        for gate_mode in ("learned", "retrieval_frozen", "both_frozen"):\n            model = moderate_model(4, gate_mode)\n            q, p, u, v, x, w, m, g, h, rho, lrho = model.quantities()\n            for lag in (1, 2, 3, 6, 20):\n                for prefix in (0, 1, 9):\n                    for a, b in ((0, 1), (1, 2)):\n                        keys = [a] * (prefix + 1) + [b] * (lag - 1)\n                        values = [-1] * prefix + [1] + [-1] * (lag - 1)\n                        loss, mass, logit = dense_stream(model.params(), model.cfg, keys, values, a)\n                        lo = boundary_log_odds(model.gaps()[a, b], m, h, rho, lag, prefix)\n                        torch.testing.assert_close(mass[prefix], torch.sigmoid(-lo), rtol=2e-12, atol=2e-14)\n                        torch.testing.assert_close(logit, -w * torch.tanh(lo / 2), rtol=2e-12, atol=2e-14)\n\n    def test_trusted_counts_path_matches_checked_path(self):\n        model = moderate_model(4)\n        counts = torch.tensor([0., 3., 1., 2., 0., 4.], dtype=torch.float64)\n        checked, checked_info = model.gradients(counts)\n        trusted, trusted_info = model.gradients(counts, validate_counts=False)\n        for expected, actual in zip(checked, trusted):\n            for x, y in zip(expected, actual):\n                torch.testing.assert_close(x, y, rtol=0, atol=0)\n        torch.testing.assert_close(checked_info["table_log_span"], trusted_info["table_log_span"])\n\n    def test_log_adam_matches_torch_adam_complete_history(self):\n        for b1, b2 in ((.9, .999), (0., .9)):\n            left = [torch.tensor([.2, -.7, .4], dtype=torch.float64)]\n            right = [left[0].clone().requires_grad_(True)]\n            log_adam = LogAdam(left, b1, b2)\n            reference = torch.optim.Adam(right, lr=.01, betas=(b1, b2), eps=1e-8)\n            for t in range(30):\n                grad = torch.tensor([math.sin(t), math.cos(t) * .001, 0.], dtype=torch.float64)\n                right[0].grad = grad.clone()\n                reference.step()\n                log_adam.step(left, [slog(grad)], [.01], math.log(1e-8))\n                torch.testing.assert_close(left[0], right[0], rtol=2e-12, atol=3e-14)\n            self.assertEqual(log_adam.t, 30)\n\n    def test_annealed_adam_matches_torch(self):\n        left = [torch.tensor([.1, .3], dtype=torch.float64)]\n        right = [left[0].clone().requires_grad_(True)]\n        optimizer = LogAdam(left, .9, .999)\n        ref = torch.optim.Adam(right, lr=.02, betas=(.9, .999), eps=.001)\n        for t in range(20):\n            eps = .001 * math.exp(-.3 * t)\n            grad = torch.tensor([(-1.) ** t * .01, math.exp(-t)], dtype=torch.float64)\n            ref.param_groups[0]["eps"] = eps\n            right[0].grad = grad\n            ref.step()\n            optimizer.step(left, [slog(grad)], [.02], math.log(eps))\n        torch.testing.assert_close(left[0], right[0], rtol=2e-12, atol=3e-14)\n\n    def test_frozen_coordinates_and_buffers(self):\n        for gate_mode in ("retrieval_frozen", "both_frozen"):\n            for kind in ("sgd", "adam"):\n                model = moderate_model(4, gate_mode)\n                before = model.theta.clone()\n                optimizer = LogAdam(model.params(), .9, .999)\n                frozen = list(model.frozen_scalar_indices)\n                for _ in range(5):\n                    gradients, _ = model.gradients()\n                    self.assertTrue(bool((gradients[-1][0][frozen] == 0).all()))\n                    rates = [.001, .001, .01]\n                    if kind == "adam":\n                        optimizer.step(model.params(), gradients, rates, math.log(1e-8))\n                    else:\n                        gradient_step(model, gradients, rates)\n                torch.testing.assert_close(before[frozen], model.theta[frozen], rtol=0, atol=0)\n                if kind == "adam":\n                    self.assertTrue(bool(torch.isneginf(optimizer.vl[-1][frozen]).all()))\n                    self.assertTrue(bool((optimizer.ms[-1][frozen] == 0).all()))\n                _, rates = rates_at(model.cfg, kind, 12)\n                self.assertTrue(bool((rates[-1][frozen] == 0).all()))\n\n    def test_initialization_and_R_adjusted_acquisition(self):\n        for R in (2, 4, 7):\n            c = Config(n=4, d=256, R=R)\n            model = Model(c)\n            self.assertFalse(torch.equal(model.Q, model.K))\n            self.assertEqual(float(model.theta[-1]), 0.)\n            self.assertAlmostEqual(float(model.quantities()[8]), c.h0)\n            self.assertEqual(float(model.theta[0]), float(model.theta[1]))\n            self.assertEqual(float(model.theta[2]), float(model.theta[3]))\n            gradients, _ = model.gradients()\n            self.assertTrue(bool((gradients[0][0] == 0).all()))\n            self.assertTrue(bool((gradients[1][0] == 0).all()))\n            self.assertTrue(bool((gradients[-1][0][:-1] == 0).all()))\n            rates = acquisition_rates(c, "sgd")\n            probe = Model(c)\n            probe.Q.zero_(); probe.K.zero_(); probe.theta[-1] = .2\n            z = torch.zeros((), dtype=torch.float64, requires_grad=True)\n            _, _, _, so, sc = probe.rows(z)\n            phi = c.wo * torch.nn.functional.softplus(-.2 * so) + c.wc * torch.nn.functional.softplus(-.2 * sc)\n            astar = -float(torch.autograd.grad(phi, z)[0]) / (c.n - 1)\n            self.assertAlmostEqual(rates[2][0], 1 / astar)\n            for rate in rates:\n                grads, _ = model.gradients()\n                gradient_step(model, grads, rate)\n            self.assertGreater(float(model.gaps()[model.mask].min()), 0.)\n            for actual, (sign, logabs) in zip(native_gradients(model)[0], model.gradients()[0]):\n                torch.testing.assert_close(actual, sign * logabs.exp(), rtol=2e-9, atol=1e-13)\n\n    def test_dataset_pair_count_and_weighted_loss(self):\n        model = moderate_model(5)\n        cases = pair_cases(model.cfg)\n        self.assertEqual(len(cases), model.cfg.n * (model.cfg.n - 1))\n        for case in cases:\n            self.assertNotEqual(case["query_key"], case["distractor_key"])\n            self.assertEqual({row["target_lag"] for row in case["recall"]}, {5})\n            self.assertEqual({row["target_lag"] for row in case["overwrite"]}, {1})\n        with tempfile.TemporaryDirectory() as path:\n            manifest = export_dataset(model.cfg, path)\n            expanded = [json.loads(line) for line in (Path(path) / "dataset.jsonl").read_text().splitlines()]\n            recall = (Path(path) / "recall_at_R.jsonl").read_text().splitlines()\n            self.assertEqual(len(recall), len(cases))\n            self.assertEqual(len(expanded), 4 * len(cases) + 2 * model.cfg.n)\n            self.assertAlmostEqual(sum(row["population_weight"] for row in expanded), 1.)\n            loss = sum(row["population_weight"] * dense_stream(model.params(), model.cfg, row["keys"],\n                                                               row["values"], row["query"], row["answer"])[0]\n                       for row in expanded)\n            torch.testing.assert_close(loss, native_loss(model), rtol=2e-12, atol=2e-14)\n\n    def test_log_backend_retains_tiny_binder_gradients(self):\n        model = moderate_model(4)\n        model.theta[2:4] = 30.\n        log_gradients, _ = model.gradients()\n        native, _ = native_gradients(model)\n        self.assertEqual(float(native[-1][2]), 0.)\n        self.assertTrue(bool(torch.isfinite(log_gradients[-1][1][2])))\n        self.assertLess(float(log_gradients[-1][1][2]), -1000.)\n        self.assertNotEqual(float(log_gradients[-1][0][2]), 0.)\n\n    def test_frozen_stale_tail_threshold(self):\n        for h in (.5, math.log(2), 1.):\n            model = Model(Config(n=3, d=5, gate_mode="retrieval_frozen", h0=h))\n            q, p, u, v, x, w, m, g, h, rho, lrho = model.quantities()\n            lo = boundary_log_odds(model.gaps()[model.mask], m, h, rho, 1, math.inf)\n            torch.testing.assert_close(torch.sigmoid(-lo), torch.full_like(lo, 1 - math.exp(-float(h))))\n            if float(h) < math.log(2):\n                self.assertTrue(bool((lo > 0).all()))\n\n    def test_config_and_counts_reject_invalid_inputs(self):\n        with self.assertRaises(ValueError):\n            Config(beta1=.99, beta2=.9)\n        with self.assertRaises(ValueError):\n            Config(R=1)\n        with self.assertRaises(ValueError):\n            Config(h0=1., x0=-1.)\n        with self.assertRaises(ValueError):\n            Config(kcal=.4, wo=.3, wc=.3)\n        model = moderate_model()\n        for counts in (torch.zeros(6), torch.ones(3), -torch.ones(6)):\n            with self.assertRaises(ValueError):\n                model.gradients(counts)\n\n\nif __name__ == "__main__":\n    unittest.main(verbosity=2)\n'


In [ ]:
SOURCES['test_runner'] = '"""Integration checks for paired training, lag evaluation and saved artifacts."""\nimport csv\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pytest\nimport torch\n\nfrom core import Config, Model\nfrom runner import evaluate, run_suite\n\n\ndef rows(path):\n    with open(path) as handle:\n        return list(csv.DictReader(handle))\n\n\ndef test_four_arms_from_identical_initialization(tmp_path):\n    torch.set_num_threads(1)\n    cfg = Config(n=4,d=64,R=4,steps=7,device="cpu",pair_batch=16)\n    result = Path(run_suite(cfg,tmp_path/"run",seeds=(3,),eval_lags=(1,4,6,7),\n                           prefixes=(0,8),log_every=5,progress=False))\n    runs = rows(result/"runs.csv")\n    assert len(runs)==4\n    assert len({r["initial_hash"] for r in runs})==1\n    assert all(r["status"]=="finite_budget_complete" for r in runs)\n    assert all(r["acquired_positive_gaps"]=="True" for r in runs)\n    hist = rows(result/"history.csv")\n    fixed = [float(r["h"]) for r in hist if r["gate_mode"]=="retrieval_frozen"]\n    assert max(fixed)==min(fixed)\n    assert np.isclose(fixed[0],cfg.h0)\n    assert all(a["passed"] for a in json.loads((result/"native_gradient_audits.json").read_text()))\n    prob = rows(result/"lag_probabilities.csv")\n    assert {int(r["lag"]) for r in prob}=={1,4,6,7}\n    assert all(0<=float(r["min_probability"])<=float(r["mean_probability"])+1e-15<=1+1e-15 for r in prob)\n    for mode in ("learned","retrieval_frozen"):\n        ck = torch.load(result/"checkpoints"/f"seed3_{mode}_adam.pt",weights_only=False)\n        assert ck["optimizer"]["t"]==cfg.steps  # history retained through acquisition\n    with pytest.raises(FileExistsError):\n        run_suite(cfg,result,seeds=(3,),progress=False)\n\n\ndef test_bound_below_finite_prefix_probability():\n    cfg = Config(n=3,d=8,R=2,device="cpu")\n    model = Model(cfg)\n    # This is a mathematical evaluation fixture, never a training initialization.\n    model.Q.zero_(); model.K.zero_()\n    model.Q[:,:3] = torch.eye(3,dtype=torch.float64)*.5\n    model.K.copy_(model.Q)\n    model.theta[0:2] = 4.\n    model.theta[-1] = 3.\n    result = evaluate(model,dict(seed=0,gate_mode="learned",optimizer="sgd"),0,0.,range(1,7),(0,1,16,128))\n    assert all(r["certified_probability_lower_bound"]<=r["min_probability"]+1e-14 for r in result)\n'


In [ ]:
SOURCES['test_asymptotics'] = '"""Independent finite-stream and numerical checks for E_t(r) envelopes."""\nimport itertools\nimport math\nimport unittest\n\nimport torch\nfrom torch.nn import functional as F\n\ntry:\n    from .asymptotics import (certified_radius, error_bounds, evaluate_asymptotics,\n                             evaluate_snapshot, scalar_error_bounds, scalar_snapshot, snapshot)\n    from .core import Config, Model, dense_stream\nexcept ImportError:\n    from asymptotics import (certified_radius, error_bounds, evaluate_asymptotics,\n                            evaluate_snapshot, scalar_error_bounds, scalar_snapshot, snapshot)\n    from core import Config, Model, dense_stream\n\n\ndef known_model(gate_mode="learned", m=20., h=1.2, w=4.):\n    model = Model(Config(n=2, d=2, R=2, gate_mode=gate_mode, h0=h))\n    model.Q = torch.tensor([[.5, .1], [0., .6]], dtype=torch.float64)\n    model.K = torch.tensor([[.5, 0.], [.05, .5]], dtype=torch.float64)\n    model.theta[:2] = math.sqrt(m)\n    model.theta[2:4] = 2.\n    model.theta[-1] = w\n    return model\n\n\nclass AsymptoticTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(1)\n\n    def test_witness_limit_matches_literal_long_streams(self):\n        model = known_model()\n        state = snapshot(model)\n        for lag in (1, 2, 3, 8):\n            with self.subTest(lag=lag):\n                finite_errors = []\n                for a, b in ((0, 1), (1, 0)):\n                    prefix = 60  # omitted geometric tail < exp(-72)\n                    keys = [a] * (prefix + 1) + [b] * (lag - 1)\n                    values = [-1] * prefix + [1] + [-1] * (lag - 1)\n                    logit = dense_stream(model.params(), model.cfg, keys, values, a)[2]\n                    finite_errors.append(float(F.logsigmoid(-logit)))\n                self.assertAlmostEqual(error_bounds(state, lag)["log_lower_error"], max(finite_errors), places=12)\n\n    def test_upper_bounds_all_short_streams(self):\n        # Enumerate arbitrary old keys/values and both labels, not just the\n        # witness family, to independently audit the uniform transfer formula.\n        model = known_model(m=6., h=1.1)\n        state = snapshot(model)\n        upper = {r: error_bounds(state, r)["upper_error"] for r in range(1, 6)}\n        alphabet = [(key, value) for key in range(2) for value in (-1., 1.)]\n        checked = 0\n        for length in range(1, 6):\n            for records in itertools.product(alphabet, repeat=length):\n                keys, values = zip(*records)\n                for query in set(keys):\n                    latest = max(index for index, key in enumerate(keys) if key == query)\n                    lag = length - latest\n                    logit = dense_stream(model.params(), model.cfg, keys, values, query, values[latest])[2]\n                    self.assertLessEqual(float(torch.sigmoid(-logit)), upper[lag] + 3e-14)\n                    checked += 1\n        self.assertGreater(checked, 2000)\n\n    def test_bounds_monotone_and_scalar_history_matches(self):\n        model = known_model()\n        state = snapshot(model)\n        previous_lower = previous_upper = -1.\n        for lag in range(1, 80):\n            row = error_bounds(state, lag)\n            scalar = scalar_error_bounds(state.delta_min, float(state.m), float(state.h),\n                                         float(state.rho), float(state.w), lag, state.max_row_norm)\n            self.assertAlmostEqual(row["log_lower_error"], scalar["log_lower_error"], places=14)\n            self.assertAlmostEqual(row["log_upper_error"], scalar["log_upper_error"], places=14)\n            self.assertLessEqual(previous_lower, row["lower_error"])\n            self.assertLessEqual(previous_upper, row["upper_error"])\n            self.assertLessEqual(row["lower_error"], row["upper_error"] + 1e-14)\n            previous_lower, previous_upper = row["lower_error"], row["upper_error"]\n\n    def test_log_errors_retain_extreme_confidence(self):\n        model = known_model(m=1e5, h=20., w=1e4)\n        model.theta[2:4] = 10.\n        row = error_bounds(snapshot(model), 1)\n        self.assertEqual(row["lower_error"], 0.)  # legitimate float64 underflow\n        self.assertEqual(row["upper_error"], 0.)\n        self.assertTrue(math.isfinite(row["log_lower_error"]))\n        self.assertLess(row["log_lower_error"], -9000.)\n        self.assertLessEqual(row["log_lower_error"], row["log_upper_error"])\n\n    def test_huge_lags_do_not_allocate_sequences_or_overflow_int64(self):\n        state = snapshot(known_model())\n        for lag in (10 ** 12, 10 ** 30, 10 ** 300):\n            row = error_bounds(state, lag)\n            self.assertEqual(row["lag"], lag)\n            self.assertAlmostEqual(row["log_lower_error"], float(F.logsigmoid(state.w)), places=14)\n            self.assertAlmostEqual(row["log_upper_error"], float(F.logsigmoid(state.w)), places=14)\n\n    def test_certified_radius_is_maximal_for_the_bound(self):\n        model = known_model(m=200., h=3., w=20.)\n        model.theta[2:4] = 4.\n        state = snapshot(model)\n        for target in (.1, .01, .001):\n            row = certified_radius(state, target)\n            radius = row["certified_radius"]\n            self.assertGreater(radius, 1)\n            self.assertLessEqual(error_bounds(state, radius)["upper_error"], target)\n            self.assertGreater(error_bounds(state, radius + 1)["upper_error"], target)\n        model.theta[-1] = 0.\n        self.assertEqual(certified_radius(snapshot(model), .1)["certified_radius"], 0)\n\n    def test_invalid_conditions_never_emit_a_zero_upper_bound(self):\n        for condition in ("norm", "gap", "decoder", "scale"):\n            model = known_model()\n            if condition == "norm":\n                model.Q *= 10\n            elif condition == "gap":\n                model.K = model.K.flip(0)\n            elif condition == "decoder":\n                model.theta[-1] = -2\n            else:\n                model.theta[0] *= -1\n            state = snapshot(model)\n            row = error_bounds(state, 2)\n            self.assertFalse(row["bound_valid"])\n            self.assertIsNone(row["upper_error"])\n            self.assertIsNone(row["log_upper_error"])\n            self.assertIsNone(certified_radius(state, .1)["certified_radius"])\n            self.assertTrue(math.isfinite(row["log_lower_error"]))\n\n    def test_predeclared_clocks_and_reference_gap(self):\n        for gate in ("learned", "retrieval_frozen"):\n            model = known_model(gate, m=10000.)\n            state = snapshot(model)\n            rows, radii = evaluate_asymptotics(model, {"seed": 8}, 900, 1e6,\n                                               reference_gap=state.delta_min / 2,\n                                               fixed_lags=(1,), theta_values=(.5,),\n                                               clock_coefficients=(.01,), error_targets=(.1,))\n            by_probe = {row["probe"]: row for row in rows}\n            power = 1 if gate == "learned" else 2\n            self.assertEqual(by_probe["clock"]["lag"], math.floor(1 + .01 * (1e6) ** power))\n            self.assertGreater(by_probe["clock"]["lag"], model.cfg.eval_max_lag)\n            self.assertTrue(by_probe["reference_theta"]["reference_gap_retained"])\n            self.assertLess(by_probe["reference_theta"]["lag"], by_probe["adaptive_theta"]["lag"])\n            self.assertEqual({row["lag"] for row in rows if row["probe"] == "fixed"}, {1, 2, 3, 4, 5})\n            self.assertEqual(radii[0]["seed"], 8)\n\n    def test_saved_scalar_history_preserves_every_probe(self):\n        model = known_model()\n        state = snapshot(model)\n        recovered = scalar_snapshot(state.delta_min, float(state.m), float(state.h),\n                                    float(state.rho), float(state.w), state.max_row_norm)\n        expected = evaluate_asymptotics(model, {"seed": 0}, 100, 200.)\n        actual = evaluate_snapshot(recovered, model.cfg.R, model.cfg.gate_mode, {"seed": 0}, 100, 200.)\n        self.assertEqual(expected, actual)\n\n\nif __name__ == "__main__":\n    unittest.main(verbosity=2)\n'


In [ ]:
SOURCES['test_a100_runner'] = '"""Portable deterministic checkpoint/resume and hardware-benchmark QA."""\nfrom dataclasses import replace\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\n\nimport torch\n\ntry:\n    from . import a100_runner as large\n    from .core import Config\nexcept ImportError:\n    import a100_runner as large\n    from core import Config\n\n\ndef options():\n    return dict(seeds=(0,), eval_lags=(1, 2, 4, 8), prefixes=(0, 8),\n                log_every=4, checkpoint_every=4, check_every=2, progress=False,\n                fixed_lags=(1, 2, 4), theta_values=(.5,), clock_coefficients=(.01,), error_targets=(.1, .01))\n\n\ndef cfg(steps=12):\n    return Config(n=4, d=32, R=2, steps=steps, device="cpu", pair_batch=16,\n                  beta1=.1, beta2=.1, max_seconds=1000.)\n\n\ndef checkpoint(directory, mode, optimizer):\n    return torch.load(Path(directory) / "checkpoints" / f"seed0_{mode}_{optimizer}.pt", map_location="cpu", weights_only=False)\n\n\ndef assert_same_state(test, left, right):\n    for a, b in zip(left["model"], right["model"]):\n        torch.testing.assert_close(a, b, rtol=0, atol=0)\n    for key in ("step", "S", "examples", "acquired", "reference_gap"):\n        test.assertEqual(left["state"][key], right["state"][key])\n    test.assertEqual(left["rng_state"], right["rng_state"])\n    test.assertEqual(left["initial_hash"], right["initial_hash"])\n    if left["optimizer"] is not None:\n        test.assertEqual(left["optimizer"]["t"], right["optimizer"]["t"])\n        for name in ("ms", "ml", "vl"):\n            for a, b in zip(left["optimizer"][name], right["optimizer"][name]):\n                torch.testing.assert_close(a, b, rtol=0, atol=0)\n\n\nclass LargeRunnerTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(1)\n\n    def test_extended_budget_exactly_matches_uninterrupted_all_four_arms(self):\n        with tempfile.TemporaryDirectory() as temporary:\n            root = Path(temporary)\n            full = large.run_large_suite(cfg(12), root / "full", **options())\n            extended = large.run_large_suite(cfg(7), root / "extended", **options())\n            large.run_large_suite(cfg(12), extended, **options())\n            for mode in ("learned", "retrieval_frozen"):\n                for optimizer in ("sgd", "adam"):\n                    left, right = checkpoint(full, mode, optimizer), checkpoint(extended, mode, optimizer)\n                    assert_same_state(self, left, right)\n                    self.assertEqual(right["state"]["status"], "finite_budget_complete")\n                    self.assertEqual(right["state"]["resume_count"], 1)\n                    self.assertTrue(all(audit["passed"] for audit in right["audits"]))\n                    self.assertEqual(len(right["audits"]), 2)\n                    self.assertTrue(right["errors"])\n                    self.assertTrue(right["radii"])\n            before = (Path(extended) / "checkpoints" / "seed0_learned_adam.pt").read_bytes()\n            large.run_large_suite(cfg(12), extended, **options())\n            self.assertEqual(before, (Path(extended) / "checkpoints" / "seed0_learned_adam.pt").read_bytes())\n            self.assertTrue((Path(extended) / "asymptotic_errors.csv").exists())\n            self.assertTrue((Path(extended) / "certified_radii.csv").exists())\n\n    def test_operational_pause_preserves_rng_and_adam_moments(self):\n        with tempfile.TemporaryDirectory() as temporary:\n            root = Path(temporary)\n            setting = {**options(), "gate_modes": ("retrieval_frozen",), "optimizers": ("adam",)}\n            full = large.run_large_suite(cfg(), root / "full", **setting)\n            resumed = large.run_large_suite(cfg(), root / "paused", max_updates=5, **setting)\n            saved = checkpoint(resumed, "retrieval_frozen", "adam")\n            self.assertEqual(saved["state"]["status"], "paused")\n            self.assertEqual(saved["state"]["step"], 5)\n            self.assertEqual(saved["optimizer"]["t"], 5)\n            large.run_large_suite(cfg(), resumed, **setting)\n            assert_same_state(self, checkpoint(full, "retrieval_frozen", "adam"), checkpoint(resumed, "retrieval_frozen", "adam"))\n\n    def test_pause_during_acquisition_preserves_whole_history(self):\n        with tempfile.TemporaryDirectory() as temporary:\n            root = Path(temporary)\n            setting = {**options(), "gate_modes": ("learned",), "optimizers": ("adam",)}\n            full = large.run_large_suite(cfg(), root / "full", **setting)\n            resumed = large.run_large_suite(cfg(), root / "paused", max_updates=1, **setting)\n            large.run_large_suite(cfg(), resumed, **setting)\n            assert_same_state(self, checkpoint(full, "learned", "adam"), checkpoint(resumed, "learned", "adam"))\n\n    def test_scientific_config_source_and_budget_guards(self):\n        with tempfile.TemporaryDirectory() as temporary:\n            setting = {**options(), "gate_modes": ("learned",), "optimizers": ("sgd",)}\n            out = large.run_large_suite(cfg(7), Path(temporary) / "run", **setting)\n            with self.assertRaisesRegex(ValueError, "Scientific configuration"):\n                large.run_large_suite(replace(cfg(12), lr_sgd=4.), out, **setting)\n            with self.assertRaisesRegex(ValueError, "shrinking"):\n                large.run_large_suite(cfg(6), out, **setting)\n            changed = large._source_hashes()\n            changed["core.py"] = "tampered"\n            with patch.object(large, "_source_hashes", return_value=changed):\n                with self.assertRaisesRegex(ValueError, "source changed"):\n                    large.run_large_suite(cfg(12), out, **setting)\n            with self.assertRaises(FileExistsError):\n                large.run_large_suite(cfg(12), out, resume=False, **setting)\n\n    def test_numerical_failure_rolls_back_all_state_to_commit(self):\n        with tempfile.TemporaryDirectory() as temporary:\n            root = Path(temporary)\n            setting = {**options(), "gate_modes": ("learned",), "optimizers": ("adam",)}\n            expected = large.run_large_suite(cfg(4), root / "expected", **setting)\n            real_step = large._optimizer_step\n\n            def corrupt(model, optimizer, gradients, rates, step):\n                real_step(model, optimizer, gradients, rates, step)\n                if step == 5:\n                    model.theta[0] = float("nan")\n\n            with patch.object(large, "_optimizer_step", side_effect=corrupt):\n                out = large.run_large_suite(cfg(12), root / "corrupted", **setting)\n            recovered = checkpoint(out, "learned", "adam")\n            self.assertEqual(recovered["state"]["status"], "numerical_stop")\n            self.assertEqual(recovered["state"]["failed_at_step"], 6)\n            assert_same_state(self, checkpoint(expected, "learned", "adam"), recovered)\n            self.assertTrue(all(bool(torch.isfinite(value).all()) for value in recovered["model"]))\n            self.assertTrue(all(row["step"] <= 4 for row in recovered["history"]))\n\n    def test_benchmark_is_fresh_and_reports_measured_local_runtime(self):\n        result = large.benchmark_config(cfg(), steps=2, warmup=1, num_seeds=2)\n        self.assertEqual(result["device"], "cpu")\n        self.assertEqual(len(result["rows"]), 4)\n        self.assertGreater(result["estimated_matrix_seconds"], 0.)\n        for row in result["rows"]:\n            self.assertEqual(row["measured_steps"], 2)\n            self.assertGreater(row["seconds_per_step"], 0.)\n            self.assertTrue(row["acquired_positive_gaps"])\n        json.dumps(result, allow_nan=False)\n\n    def test_compiled_gradient_audit_has_no_silent_fallback(self):\n        model = large.Model(cfg())\n        model.theta[-1] = .2\n        backend = large._GradientBackend(model, compiled=True)\n        with patch.object(torch, "compile", side_effect=RuntimeError("test compile failure")):\n            with self.assertRaises(RuntimeError):\n                backend()\n\n\nif __name__ == "__main__":\n    unittest.main(verbosity=2)\n'


In [ ]:
import hashlib
import importlib
import inspect
import json
import subprocess
from dataclasses import asdict

for name, source in SOURCES.items():
    (CODE_DIR / (name + ".py")).write_text(source, encoding="utf-8")
    sys.modules.pop(name, None)
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
from core import Config
from a100_runner import run_large_suite, benchmark_config
from report import make_report
from asymptotic_report import make_asymptotic_report

SOURCE_HASHES = {name + ".py": hashlib.sha256(source.encode()).hexdigest()
                 for name, source in SOURCES.items()}
print("Embedded source SHA-256:", json.dumps(SOURCE_HASHES, indent=2))
if RUN_QA:
    if importlib.util.find_spec("pytest") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pytest>=7"])
    tests = [str(CODE_DIR / (name + ".py")) for name in SOURCES if name.startswith("test_")]
    subprocess.check_call([sys.executable, "-m", "pytest", "-q", *tests], cwd=CODE_DIR,
                          env=dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1"))

cfg = Config(n=N, d=WIDTH, R=R, steps=STEPS, device=DEVICE, h0=H0,
             sigma=1e-6, q0=0.5, u0=1.7, beta1=BETA1, beta2=BETA2,
             lr_sgd=5.0, lr_adam=0.015, offset=1000.0, power=0.75,
             table_lr=1e-9, table_power=2.0, pair_batch=4096, batch_growth=0.5,
             eps_decay=0.1, max_seconds=MAX_BRANCH_SECONDS)
GATE_MODES, OPTIMIZERS = ("learned", "retrieval_frozen"), ("sgd", "adam")
dimension_bound = int(np.ceil(144*np.log(16*N*(N-1)/0.05)))
print(json.dumps(asdict(cfg), indent=2))
print("Width:", WIDTH, "| stated Gaussian dimension bound at failure budget 0.05:", dimension_bound,
      "| satisfied:", WIDTH >= dimension_bound)
print("Frozen h0 > log(2):", H0 > np.log(2))


## Measured seed-0 throughput before the long run

The benchmark uses independent seed-0 models and their acquisition protocol. Its updates are
discarded; the actual experiment starts from the original draws or an exact saved checkpoint.
It measures the requested model size on the detected device and estimates all requested arms.
Later optimizer states, growing SGD batches, certificate evaluations, checkpoint writes and
Drive latency can change runtime. The estimate is not a promised completion time.


In [ ]:
BENCHMARK = None
if COMPILE:
    print("Experimental compilation requested. The runner must pass its eager/compiled signed-log gradient audit; failures raise an error.")
if RUN_BENCHMARK and RUN_TRAINING:
    BENCHMARK = benchmark_config(cfg, steps=BENCHMARK_STEPS, warmup=BENCHMARK_WARMUP,
                                 num_seeds=len(SEEDS), compile_gradients=COMPILE)
    print("Measured preflight:", json.dumps(BENCHMARK, indent=2))
else:
    print("Throughput benchmark skipped by settings.")


## Run, checkpoint, and resume

The stable default tag is `fox_a100_asymptotics_v1`, followed by the moment-preset suffix.
Leave the tag and training/evaluation settings unchanged to resume after a disconnect.
Checkpoints include model parameters, signed-log Adam state, RNG state, continuation clock and
progress. Saved rows and run statuses are retained. Completed arms are skipped on resume.
Use a **new tag** when changing protocol settings; incompatible checkpoints are rejected.

By default, diagnostics are collected every 1,000 updates and committed with resumable
checkpoints every 5,000 updates. With Drive enabled,
the files live outside the temporary runtime. A disconnect can lose work since the latest
checkpoint (up to 5,000 updates by default); it does not prove a branch converged or failed
theoretically. Inspect `runs.csv`.
For a first full seed set `SEEDS=(0,)`; run the additional seeds as a separate tag if the saved
manifest requires the original seed list to remain unchanged.


In [ ]:
if RUN_TRAINING:
    OUT_DIR = Path(run_large_suite(
        cfg, OUT_DIR, seeds=SEEDS, gate_modes=GATE_MODES, optimizers=OPTIMIZERS,
        eval_lags=EVAL_LAGS, prefixes=PREFIXES, log_every=LOG_EVERY,
        checkpoint_every=CHECKPOINT_EVERY, resume=RESUME,
        compile_gradients=COMPILE,
        fixed_lags=FIXED_LAGS, theta_values=THETA_VALUES,
        clock_coefficients=CLOCK_COEFFICIENTS, error_targets=ERROR_TARGETS,
    ))
else:
    print("Training skipped; using saved output folder:", OUT_DIR)

if not OUT_DIR.exists():
    raise FileNotFoundError(f"No output at {OUT_DIR}. Run training or set the saved run tag.")
embedded = OUT_DIR / "embedded_source"
embedded.mkdir(exist_ok=True)
for name, source in SOURCES.items():
    (embedded / (name + ".py")).write_text(source, encoding="utf-8")
(embedded / "sha256.json").write_text(json.dumps(SOURCE_HASHES, indent=2), encoding="utf-8")
(OUT_DIR / "notebook_hardware.json").write_text(json.dumps(HARDWARE, indent=2), encoding="utf-8")
if BENCHMARK is not None:
    (OUT_DIR / "notebook_benchmark.json").write_text(json.dumps(BENCHMARK, indent=2), encoding="utf-8")


## Eₜ(r), Eₜ(rₜ), and the actual radius trajectories

The main plots show fixed and moving radius brackets against both update count and each run's
own continuation learning-rate sum S. The next plot shows the actual rₜ values and the certified
radius at each error threshold. Read the interval endpoints as bounds on the unknown supremum.
Crosses at log10(error)=0 explicitly mark unavailable uniform certificates.

The optimizer panel compares q/S, u/S and w/S with their nominal Adam multipliers and shows
the current signed normalized growth directions separately. This helps distinguish slow moment
tracking from failure of the predicted asymptotic geometry. Its ratios include acquisition offsets,
and the displayed references remain conditional asymptotic values.

The ordinary lag plot remains useful for finite empirical behavior, including cases where Adam
already generalized beyond some tested lag at an earlier checkpoint. A final fixed-lag curve
alone does not distinguish bounded from expanding asymptotic generalization.


In [ ]:
ASYMPTOTIC_REPORT = make_asymptotic_report(OUT_DIR)
ORDINARY_REPORT = make_report(OUT_DIR)
for name in ("asymptotic_fixed_error_by_step", "asymptotic_fixed_error_by_S",
             "asymptotic_moving_error_by_step", "asymptotic_moving_error_by_S",
             "radius_trajectories", "optimizer_diagnostics"):
    display(Image(filename=ASYMPTOTIC_REPORT[name], width=1200))
display(Image(filename=ORDINARY_REPORT["generalization_by_lag"], width=1200))
display(Image(filename=ORDINARY_REPORT["theory_diagnostics"], width=1400))
display(Markdown(Path(ASYMPTOTIC_REPORT["report"]).read_text(encoding="utf-8")))
print("All reports:", {"asymptotic": ASYMPTOTIC_REPORT, "finite_prefix": ORDINARY_REPORT})


## Retain results and checkpoints

Drive outputs are already persistent. The optional ZIP contains CSV measurements, certificates,
reports, exact source and resumable checkpoints; it can be large for the full profile.
No local A100 result is preloaded or fabricated by this notebook. Its displayed evidence comes
only from the runs you execute or explicitly resume.


In [ ]:
if DOWNLOAD_AT_END:
    import shutil
    archive_parent = Path("/content") if IN_COLAB else Path(tempfile.gettempdir())
    zip_path = Path(shutil.make_archive(str(archive_parent / EFFECTIVE_RUN_TAG), "zip",
                                       root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name))
    if IN_COLAB:
        from google.colab import files
        files.download(str(zip_path))
    else:
        display(FileLink(str(zip_path)))
else:
    print("Measurements, reports and checkpoints retained at:", OUT_DIR)
